# Semana 12 — Python com PostgreSQL para ETL

Nesta semana você constrói, do zero ao fim, um **pipeline ETL** completo: um programa em Python que lê os dados do banco de produção da Northwind, organiza esses dados em camadas de qualidade crescente (Bronze, Prata e Ouro), carrega o resultado de volta no PostgreSQL e — o passo que quase todo mundo esquece — **confere se o que chegou lá está correto**.

**⚠️ Este notebook precisa rodar no VS Code local (Windows), não no Google Colab.** O motivo é o mesmo da Semana 11: o PostgreSQL está instalado no *seu* computador, e o Google Colab roda numa máquina na nuvem, que não enxerga o `localhost` da sua máquina.

São 3 encontros nesta semana, e o notebook está dividido exatamente assim:

| Encontro | O que você constrói |
|---|---|
| **Dia 1** | Extração (o **E** de *Extract*) e a camada **Bronze** — o dado bruto, fiel à origem |
| **Dia 2** | Transformação (o **T** de *Transform*) e a camada **Prata** — o dado limpo e confiável |
| **Dia 3** | Camada **Ouro**, gráficos, carga no banco (o **L** de *Load*), validação e controle de falhas — e, no final, o banco aberto dentro do VS Code e a **entrega do analista**, com o painel de gráficos |

### 🧭 De onde você vem

**O que você viu na Semana 11:** a integração entre Python e SQL. Você aprendeu que um programa Python pode abrir uma conexão com um banco de dados e disparar comandos SQL por conta própria, sem ninguém abrir uma ferramenta de banco manualmente — primeiro com a biblioteca `pyodbc` no SQLite, depois com a biblioteca `psycopg2` no PostgreSQL.

**O que você fez:** o CRUD completo por código — `INSERT` para criar registros, `SELECT` para ler, `UPDATE` para alterar e `DELETE` para remover. Você também usou `pd.read_sql()`, que executa uma consulta SQL e já devolve o resultado como DataFrame do Pandas, pronto para analisar. E fechou a semana com um `ALTER TABLE ... RENAME`, traduzindo as tabelas e colunas principais do banco `northwind` para português.

**O que você estudou:** o banco `northwind`, que é o sistema da Northwind Traders — uma importadora e distribuidora de alimentos. É um banco de dados real, com 830 pedidos feitos entre 1996 e 1998, 91 clientes espalhados por 21 países e 77 produtos em catálogo. Por causa da tradução que você mesmo rodou, as tabelas hoje se chamam `clientes`, `pedidos`, `itens_pedido`, `produtos` e `categorias`.

**Qual era o objetivo:** fazer o Python conversar com o banco de dados. Até a Semana 10, o SQL vivia isolado numa ferramenta; a partir da Semana 11, o SQL virou parte do seu código.

**Onde você está agora:** o `northwind` é o **banco transacional** da Northwind — o sistema que registra cada venda no momento em que ela acontece. Nesta semana você para de olhar esse banco como "um lugar de onde eu leio dados" e passa a tratá-lo como a **origem de um processo**: um pipeline que extrai, limpa, agrega, carrega e valida. É esse processo que transforma um banco de operação num banco de análise.

### 🟢 O que é um ETL, e por que ele existe

**ETL** é a sigla de três etapas, nessa ordem:

- **E — Extract (extrair):** trazer o dado da origem para dentro do seu programa.
- **T — Transform (transformar):** limpar, padronizar, corrigir e enriquecer esse dado.
- **L — Load (carregar):** gravar o resultado no destino, onde a análise vai acontecer.

A pergunta natural é: se o dado já está num banco de dados, por que não analisar direto ali? Por dois motivos concretos.

O primeiro é que o banco transacional é feito para **registrar** vendas, não para **analisar** vendas. Ele guarda cada informação numa tabela separada (cliente numa, produto em outra, item de pedido numa terceira) porque isso deixa o cadastro rápido e sem repetição. Só que toda análise precisa juntar essas tabelas de novo — e rodar esse `JOIN` pesado várias vezes por dia, no mesmo banco que está atendendo as vendas, deixa o sistema de vendas lento.

O segundo é que o dado bruto quase nunca está pronto. No `northwind` existem pedidos sem data de envio, produtos que a empresa não vende mais, campos de região vazios e datas que o Python ainda nem reconhece como datas. Analisar isso sem tratamento produz um número errado com cara de número certo — que é o pior tipo de erro que existe numa análise.

**A arquitetura medalhão** é a forma mais usada de organizar esse processo. Em vez de ir do dado bruto direto para o relatório, você passa por três camadas, e cada uma tem uma responsabilidade só:

| Camada | O que guarda | Regra de ouro |
|---|---|---|
| 🥉 **Bronze** | O dado bruto, exatamente como veio da origem | Nada é filtrado nem corrigido aqui |
| 🥈 **Prata** | O dado limpo, padronizado e com regras de negócio aplicadas | Toda decisão de limpeza fica documentada |
| 🥇 **Ouro** | O dado agregado, pronto para virar relatório ou gráfico | Ninguém consulta linha a linha aqui |

O nome "medalhão" vem justamente da ordem bronze → prata → ouro: o dado vai ficando mais valioso a cada camada. Você vai construir as três, uma por vez, e vai enxergar cada uma delas de duas formas: como **arquivo, no Explorer do VS Code**, e como **tabela, dentro do PostgreSQL**.

---

# 📅 DIA 1 — Extração e camada Bronze

Neste primeiro encontro você conecta ao banco de produção, entende exatamente o que existe lá dentro, extrai os dados de venda e guarda essa extração como camada Bronze.

## Passo 0 — Preparando o ambiente

Este passo prepara o computador para rodar o notebook. Faça com calma: quase todo problema da semana nasce de um ambiente mal preparado, e não do código.

### 0.1 — Escolhendo o Python que vai rodar o notebook (o kernel)

Um notebook não roda sozinho. Quem executa cada célula é um **kernel**: o Python instalado no seu computador, ligado ao notebook. Antes da primeira célula, o VS Code precisa saber qual Python usar.

1. Com este notebook aberto, olhe o **canto superior direito** da tela. Aparece um botão escrito **Select Kernel** (se já houver um Python escolhido, aparece o nome dele, como `Python 3.12`).
2. Clique nele. Na lista que abre no topo da tela, escolha **Python Environments**.
3. Escolha o Python instalado na sua máquina (o nome começa com `Python 3`). Se aparecer mais de um, prefira o que tem a versão mais recente.
4. Se o VS Code sugerir instalar as extensões **Python** e **Jupyter**, clique em **Install** — sem elas o notebook não roda.

**Na primeira célula que você executar**, pode aparecer um aviso no canto inferior direito: *"Running cells with 'Python 3...' requires the ipykernel package"*. O `ipykernel` é a peça que liga o Python ao notebook. Clique em **Install** e espere a instalação terminar; a célula roda em seguida.

**Como executar uma célula:** clique dentro dela e pressione **Shift + Enter**, ou clique no triângulo ▶ que aparece à esquerda da célula. Enquanto a célula roda, aparece um relógio no lugar do triângulo; quando termina, surge um número entre colchetes, como `[1]`, que indica a ordem em que as células foram executadas.

### 0.2 — Instalando as bibliotecas

Três bibliotecas são necessárias nesta semana:

- **`psycopg2`** — é o tradutor entre o Python e o PostgreSQL. É ele que abre a conexão e leva os comandos SQL até o banco. Você já usou essa biblioteca no bônus da Semana 11. A versão que se instala é a `psycopg2-binary`, que já vem compilada e não exige nenhum programa extra no Windows.
- **`pandas`** — a biblioteca de manipulação de tabelas que você usa desde a Semana 05. É ela que vai receber o resultado das consultas em formato de DataFrame.
- **`matplotlib`** — a biblioteca de gráficos da Semana 07. Ela só entra no Dia 3, mas instalar tudo de uma vez agora evita interromper a aula depois.

A célula abaixo instala as três, e também o `ipykernel` do passo anterior — se ele já estiver instalado, nada muda. O `%pip install` é um comando do próprio Jupyter/VS Code: ele chama o instalador de pacotes do Python sem você precisar sair do notebook e abrir um terminal separado.

In [1]:
%pip install psycopg2-binary pandas matplotlib ipykernel

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ------------------------- -------------- 1.8/2.9 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 9.3 MB/s  0:00:00
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.5 MB 8.5 MB/s eta 0:00:01
   ----------- ---------------------------- 2.6/9.5 MB 6.7 MB/s eta 0:00:02
   ------------ --------------------------- 2.9/9.5 MB 5.0 MB/s eta 0:00:02
   -------------- ------------------------- 3.4/9.5 MB 3.7 MB/s eta 0:00:02
   ---------------- ----------------------- 3.9/9.5 MB 3.3 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.5 MB 3.5 MB/s eta 0:00:02
   ------------------------- -------------- 6.0/9.5 MB 3.8 MB/s eta 0:00:01
   --------------------------- ------------ 6.6/9.5

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Como saber se a instalação deu certo.** A célula mostra várias linhas de texto. As que importam são as últimas:

- `Successfully installed ...` — a biblioteca foi instalada agora.
- `Requirement already satisfied ...` — a biblioteca já estava instalada. Também está certo.

Se a última linha falar em `ERROR`, confira a internet do computador e rode a célula de novo.

**Instalou, mas a próxima célula diz `ModuleNotFoundError: No module named ...`?** O kernel que já estava aberto antes da instalação ainda não enxerga a biblioteca nova. Clique em **Restart**, na barra no topo do notebook, espere o reinício e rode a célula de novo. Não é preciso instalar outra vez.

**Uma regra para a semana inteira:** o Dia 2 e o Dia 3 começam com esta mesma célula de instalação. Cada encontro abre o VS Code do zero, às vezes em outro computador do laboratório, e rodar a instalação de novo nunca faz mal.

### 🔑 Antes de rodar qualquer coisa: a sua senha

Em todas as células deste notebook a conexão aparece com `password="SUA_SENHA_AQUI"`. Esse é um texto de exemplo, não uma senha de verdade — você precisa trocar pela senha que definiu quando instalou o PostgreSQL na sua máquina.

Para não fazer isso célula por célula, use o localizar e substituir do VS Code: pressione **Ctrl + H**, escreva `SUA_SENHA_AQUI` no primeiro campo, a sua senha no segundo, e clique no ícone de **substituir tudo** (ou pressione **Ctrl + Alt + Enter**). Todas as células passam a usar a sua senha de uma vez só.

Uma observação profissional sobre isso: escrever senha dentro do código é aceitável aqui porque este notebook é um exercício de aula que nunca sai da sua máquina. Em um projeto real, a senha nunca fica escrita no arquivo — ela vem de uma variável de ambiente ou de um arquivo de configuração separado, justamente para não ir parar no GitHub junto com o código.

### 📖 Antes do código — abrindo a conexão

O código abaixo faz três coisas, e vale entender cada uma antes de rodar:

`psycopg2.connect(...)` abre a conexão com o PostgreSQL e devolve um objeto de conexão. Os quatro parâmetros dizem **onde** o banco está (`host="localhost"` significa "nesta mesma máquina"), **qual** banco você quer (`dbname="northwind"`), e **quem** está entrando (`user` e `password`).

`conexao.cursor()` cria o cursor, que é o objeto que efetivamente executa comandos SQL e busca resultados. A conexão é a linha telefônica; o cursor é a conversa.

`cursor.execute("SELECT version()")` manda um comando SQL para o banco, e `cursor.fetchone()` traz de volta a primeira (e neste caso única) linha do resultado. `SELECT version()` é uma consulta que não depende de nenhuma tabela: ela só devolve qual versão do PostgreSQL está rodando. É por isso que ela serve como teste — se essa consulta responder, a conexão está funcionando.

`conexao.close()` encerra a conexão. Toda conexão aberta consome um recurso do servidor, então fechar ao terminar é obrigatório, não opcional.

In [1]:
import psycopg2

conexao = psycopg2.connect(
    host="localhost",
    port="5433",
    dbname="northwind",
    user="postgres",
    password="admin",
)
cursor = conexao.cursor()

cursor.execute("SELECT version()")
print("Conexão bem sucedida!")
print(cursor.fetchone()[0])

conexao.close()

Conexão bem sucedida!
PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit


### ⚠️ Um erro real: senha incorreta

Antes de seguir, veja como o Python reclama quando a senha está errada — porque esse é, com folga, o erro mais comum desta semana.

A célula abaixo tenta conectar de propósito com a senha `senha_errada_de_proposito`. O `try`/`except` captura o erro para o notebook não parar: sem ele, a célula lançaria a exceção e todas as células seguintes ficariam sem rodar.

Repare que o `except` captura `Exception`, que é a classe mãe de todos os erros do Python — e não um tipo específico. A razão para essa escolha está logo abaixo do código, e ela é mais interessante do que parece.

`type(erro).__name__` devolve o nome da classe do erro que aconteceu. É assim que você descobre, na prática, **qual** erro o Python levantou.

In [3]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="northwind",
        user="postgres",
        password="senha_errada_de_proposito",
    )
    print("Conectou (não deveria!)")
except Exception as erro:
    print("Deu erro, e o tipo dele é:", type(erro).__name__)
    print("Mensagem:", erro)

Deu erro, e o tipo dele é: OperationalError
Mensagem: connection to server at "localhost" (::1), port 5432 failed: FATAL:  password authentication failed for user "postgres"



Se o seu PostgreSQL foi instalado em **português**, o tipo de erro que apareceu foi `UnicodeDecodeError`, com uma mensagem estranha sobre não conseguir decodificar um byte:

```
'utf-8' codec can't decode byte 0xe7 in position 78: invalid continuation byte
```

Isso confunde qualquer pessoa na primeira vez, então vale entender o que aconteceu. O servidor respondeu `FATAL: a autenticação do usuário "postgres" falhou` — em português, com acento. O byte `0xe7` é justamente o **ç** de "autenticação". O `psycopg2` tenta ler essa resposta como UTF-8, não consegue por causa do acento, e o erro de leitura acontece **antes** de a biblioteca conseguir montar o erro de conexão que ela mandaria normalmente.

Em resumo: a mensagem fala de codificação, mas o problema real é a conexão.

**A regra prática que você leva desta célula:** qualquer erro que aconteça dentro do `psycopg2.connect()` — seja `UnicodeDecodeError`, seja `OperationalError` — significa uma destas três coisas:

| Causa | Como confirmar |
|---|---|
| Senha errada | é a causa mais provável; confira a senha do PostgreSQL |
| Banco inexistente | confira a grafia de `dbname`, e se o `northwind` foi restaurado nesta máquina |
| Servidor desligado | abra os Serviços do Windows e veja se o serviço `postgresql` está em execução |

É por isso que o `except` desta célula captura `Exception` em vez de `psycopg2.OperationalError`: em máquina com PostgreSQL em inglês viria `OperationalError`, em máquina com PostgreSQL em português vem `UnicodeDecodeError`, e capturar a classe mãe funciona nos dois casos.

## Passo 1 — Conhecendo a origem

Antes de extrair qualquer coisa, um princípio que separa quem faz ETL de quem só move dado: **você precisa saber o que existe na origem antes de tirar algo de lá**. Extrair uma tabela sem saber quantas linhas ela tem, ou quais colunas aceitam valor vazio, é como assinar um contrato sem ler.

O PostgreSQL guarda essa informação sobre si mesmo num conjunto de tabelas especiais chamado `information_schema`. É um catálogo: um lugar onde o banco anota quais tabelas existem, quais colunas cada uma tem, e de que tipo é cada coluna.

### 📖 Antes do código — `pd.read_sql()` e o catálogo

`pd.read_sql(consulta, conexao)` executa a consulta SQL no banco e devolve o resultado **já como um DataFrame do Pandas**, com as colunas nomeadas. É a ponte entre o mundo SQL e o mundo Pandas — e, a partir de agora, é assim que você vai trazer praticamente todo dado para dentro do Python.

A consulta abaixo lê a tabela `information_schema.tables`, filtrando por `table_schema = 'public'` (o espaço onde ficam as tabelas do dia a dia, não as internas do PostgreSQL) e por `table_type = 'BASE TABLE'` (tabelas de verdade, excluindo as *views*, que você criou na Semana 09).

`display()` é a função do Jupyter que mostra um DataFrame como tabela formatada, com as bordas e o cabeçalho — diferente do `print()`, que mostraria o texto cru.

**⚠️ Um aviso amarelo vai aparecer, e ele é normal.** Ao rodar a célula, o Pandas mostra esta mensagem:

```
UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or
database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested.
```

Traduzindo: o Pandas avisa que ele testa oficialmente só conexões feitas com a biblioteca SQLAlchemy, e que a sua conexão (feita com `psycopg2`) está fora dessa lista. Repare na palavra **aviso**: não é erro. O código roda, o resultado vem correto, e esse aviso vai reaparecer em toda `pd.read_sql()` deste notebook — pode ignorar.

O SQLAlchemy é uma biblioteca que funciona como uma camada por cima do `psycopg2`, com recursos extras que este curso não usa. Para tudo que você vai fazer nesta semana, o `psycopg2` sozinho dá conta.

In [4]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    port="5433",
    dbname="northwind",
    user="postgres",
    password="admin",
)

tabelas = pd.read_sql('''
    SELECT table_name AS tabela
    FROM information_schema.tables
    WHERE table_schema = 'public'
      AND table_type = 'BASE TABLE'
    ORDER BY table_name
''', conexao)

print("Tabelas encontradas no banco northwind:", len(tabelas))
display(tabelas)

conexao.close()

Tabelas encontradas no banco northwind: 16


C:\Users\laisa\AppData\Local\Temp\ipykernel_19676\913834342.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tabelas = pd.read_sql('''


,tabela
0,categorias
1,clientes
2,customer_customer_demo
3,customer_demographics
4,departamentos
5,employee_territories
6,employees
7,funcionarios
8,itens_pedido
9,pedidos


Repare no resultado: existem tabelas com nome em português (`clientes`, `pedidos`, `itens_pedido`, `produtos`, `categorias`) e tabelas com nome em inglês (`employees`, `suppliers`, `shippers`, `territories`, `region`).

Essa mistura tem uma explicação, e é a sua própria: na Semana 11 você rodou um `ALTER TABLE ... RENAME` que traduziu **apenas** as cinco tabelas mais usadas nas consultas de venda. As demais continuaram com o nome original.

Isso não é um defeito do exercício — é exatamente o tipo de coisa que você encontra num banco real. Sistemas antigos acumulam padrões de nomes diferentes ao longo dos anos, conforme equipes diferentes mexem neles. Um ETL bem feito lida com a origem como ela é, e entrega um destino padronizado. É justamente esse o trabalho desta semana.

### 📖 Antes do código — contando as linhas de cada tabela

A próxima célula usa `COUNT(*)`, a função de agregação do SQL que conta quantas linhas existem numa tabela. Ela aparece aqui dentro de uma consulta com `UNION ALL`, que empilha o resultado de várias consultas numa tabela só — cada `SELECT` produz uma linha, e o `UNION ALL` junta todas.

Também aparece a sintaxe `'clientes' AS tabela`: quando você escreve um texto entre aspas simples no `SELECT`, o SQL cria uma coluna com aquele valor fixo em todas as linhas. É assim que cada linha do resultado ganha o nome da tabela que ela está contando.

In [5]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    port="5433",
    dbname="northwind",
    user="postgres",
    password="admin",
)

volumes = pd.read_sql('''
    SELECT 'clientes' AS tabela, COUNT(*) AS linhas FROM clientes
    UNION ALL
    SELECT 'pedidos', COUNT(*) FROM pedidos
    UNION ALL
    SELECT 'itens_pedido', COUNT(*) FROM itens_pedido
    UNION ALL
    SELECT 'produtos', COUNT(*) FROM produtos
    UNION ALL
    SELECT 'categorias', COUNT(*) FROM categorias
    ORDER BY linhas DESC
''', conexao)

display(volumes)

conexao.close()

C:\Users\laisa\AppData\Local\Temp\ipykernel_19676\3638061773.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  volumes = pd.read_sql('''


,tabela,linhas
0,itens_pedido,2155
1,pedidos,830
2,clientes,91
3,produtos,77
4,categorias,8


Esses números contam uma história, e vale ler com atenção: são **830 pedidos**, mas **2155 itens de pedido**. Ou seja, cada pedido tem em média 2,6 produtos diferentes dentro dele.

Essa diferença é o conceito de **grão** (ou granularidade): o que representa **uma linha** de uma tabela. Na tabela `pedidos`, uma linha é um pedido inteiro. Na tabela `itens_pedido`, uma linha é um produto dentro de um pedido.

Guarde isso, porque é a origem do erro de análise mais comum que existe: contar 2155 e dizer "a Northwind teve 2155 pedidos". Ela teve 830. No Passo 2 você vai ver esse efeito acontecendo na prática.

### ✏️ Atividade Prática 1 — Sua vez de programar

**Contextualização:** antes de aprovar o projeto do novo painel de vendas, a diretoria da Northwind quer uma resposta simples do time de dados: em que período os dados do sistema realmente começam e terminam? Ninguém quer descobrir, depois do painel pronto, que o banco só tem dados de um ano e meio — e que a comparação "ano contra ano" que a diretoria pediu é impossível.

**Comando:** conecte no banco `northwind` e escreva uma consulta na tabela `pedidos` que traga, numa única linha, três informações: a data do pedido mais antigo (`MIN(data_pedido)`), a data do pedido mais recente (`MAX(data_pedido)`) e o total de pedidos (`COUNT(*)`). Dê um nome a cada coluna do resultado usando `AS` (por exemplo, `AS primeiro_pedido`). Mostre o resultado com `display()` e feche a conexão ao final.

In [8]:
# escreva seu código aqui

import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    port="5433",
    dbname="northwind",
    user="postgres",
    password="admin",
)

info_pedidos = pd.read_sql('''
    SELECT MIN(data_pedido) AS primeiro_pedido,
           MAX(data_pedido) AS ultimo_pedido, 
           COUNT(*) AS total FROM pedidos
''', conexao)

display(info_pedidos)

conexao.close()

C:\Users\laisa\AppData\Local\Temp\ipykernel_19676\450876848.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  info_pedidos = pd.read_sql('''


,primeiro_pedido,ultimo_pedido,total
0,1996-07-04,1998-05-06,830


## Passo 2 — Extract: trazendo o dado para o Python

Chegou o **E** do ETL. Extrair, aqui, significa executar uma consulta que junta as tabelas de venda e trazer esse resultado para dentro de um DataFrame.

A decisão importante deste passo é **o que extrair**. A regra da camada Bronze é clara: traga tudo que for relevante para a análise, **sem filtrar nada**. Nada de excluir pedidos sem data de envio, nada de remover produtos descontinuados — esses recortes existem, e você vai fazer todos eles, mas na camada Prata, no Dia 2. A Bronze precisa ser uma cópia fiel da origem, porque ela é a sua rede de segurança: se uma regra de limpeza estiver errada, você reconstrói a Prata a partir da Bronze, sem precisar incomodar o banco de produção de novo.

### 📖 Antes do código — a consulta de extração

A consulta abaixo junta cinco tabelas com `JOIN`, exatamente como você aprendeu na Semana 09. A tabela que comanda é `itens_pedido`, e a razão é o grão: você quer uma linha por **item vendido**, que é o nível mais detalhado que existe. A partir dela, cada `JOIN` acrescenta o contexto que falta.

| Junção | O que acrescenta |
|---|---|
| `JOIN pedidos` | a data do pedido, a data de envio e o frete |
| `JOIN clientes` | quem comprou, de qual cidade e país |
| `JOIN produtos` | o nome do produto e se ele ainda é vendido |
| `JOIN categorias` | a categoria à qual o produto pertence |

Duas coisas na sintaxe que valem reforço da Semana 09: os apelidos (`ip`, `p`, `cl`, `pr`, `cat`) encurtam o nome da tabela dentro da consulta, e o `AS` renomeia uma coluna no resultado — aqui usado para deixar claro que `pais_cliente` vem do cadastro do cliente, e não do endereço de entrega do pedido.

No fim do código aparece `bronze.head()`, método que você usa desde a Semana 06: ele mostra as primeiras linhas do DataFrame (5, por padrão), para você conferir a cara do resultado sem despejar 2155 linhas na tela. O Passo 3 retoma esse e outros métodos de inspeção com calma.

In [9]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    port="5433",
    dbname="northwind",
    user="postgres",
    password="admin",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido,
           p.data_pedido,
           p.required_date AS data_prometida,
           p.data_envio,
           p.freight AS frete,
           p.ship_country AS pais_entrega,
           p.ship_region AS regiao_entrega,
           cl.id_cliente,
           cl.nome_empresa AS cliente,
           cl.cidade AS cidade_cliente,
           cl.pais AS pais_cliente,
           pr.nome_produto AS produto,
           pr.descontinuado,
           cat.nome_categoria AS categoria,
           ip.preco_unitario,
           ip.quantidade,
           ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)

print("Linhas extraídas:", len(bronze))
print("Colunas extraídas:", len(bronze.columns))
display(bronze.head())

conexao.close()

Linhas extraídas: 2155
Colunas extraídas: 17


C:\Users\laisa\AppData\Local\Temp\ipykernel_19676\899322793.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  bronze = pd.read_sql('''


,id_pedido,data_pedido,data_prometida,data_envio,frete,pais_entrega,regiao_entrega,id_cliente,cliente,cidade_cliente,pais_cliente,produto,descontinuado,categoria,preco_unitario,quantidade,desconto
0,10248,1996-07-04,1996-08-01,1996-07-16,32.38,France,NaN,VINET,Vins et alcools Chevalier,Reims,France,Queso Cabrales,0,Dairy Products,14.0,12,0.0
1,10248,1996-07-04,1996-08-01,1996-07-16,32.38,France,NaN,VINET,Vins et alcools Chevalier,Reims,France,Singaporean Hokkien Fried Mee,1,Grains/Cereals,9.8,10,0.0
2,10248,1996-07-04,1996-08-01,1996-07-16,32.38,France,NaN,VINET,Vins et alcools Chevalier,Reims,France,Mozzarella di Giovanni,0,Dairy Products,34.8,5,0.0
3,10249,1996-07-05,1996-08-16,1996-07-10,11.61,Germany,NaN,TOMSP,Toms Spezialitäten,Münster,Germany,Tofu,0,Produce,18.6,9,0.0
4,10249,1996-07-05,1996-08-16,1996-07-10,11.61,Germany,NaN,TOMSP,Toms Spezialitäten,Münster,Germany,Manjimup Dried Apples,0,Produce,42.4,40,0.0


São **2155 linhas e 17 colunas**. E esse 2155 é exatamente o número da tabela `itens_pedido` que você contou no Passo 1 — não é coincidência: o `JOIN` partiu de `itens_pedido` e cada uma das outras tabelas contribuiu apenas com contexto, sem criar nem perder linhas.

### 📖 Antes do código — vendo as colunas que você acabou de criar

Toda vez que uma camada nasce, duas perguntas precisam de resposta: **quais colunas ela tem** e **de onde cada coluna veio**. O código abaixo responde as duas.

`bronze.columns` devolve a lista com o nome de todas as colunas do DataFrame. Encadeando `.tolist()`, essa lista vira uma lista comum do Python, mais fácil de percorrer com um `for`.

`enumerate(lista, start=1)` percorre a lista devolvendo, a cada volta, a posição e o valor — com `start=1` a contagem começa em 1 em vez de 0, que é o que faz sentido para uma pessoa ler.

`bronze.dtypes` devolve o tipo de dado de cada coluna.

`pd.DataFrame({...})` cria um DataFrame novo a partir de um dicionário: cada chave vira o nome de uma coluna, e a lista associada a ela vira o conteúdo daquela coluna. É assim que o código monta, do zero, a tabela que mostra nome, tipo e origem de cada coluna da camada Bronze.

In [13]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    port="5433",
    dbname="northwind",
    user="postgres",
    password="admin",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, p.data_pedido, p.required_date AS data_prometida, p.data_envio,
           p.freight AS frete, p.ship_country AS pais_entrega, p.ship_region AS regiao_entrega,
           cl.id_cliente, cl.nome_empresa AS cliente, cl.cidade AS cidade_cliente,
           cl.pais AS pais_cliente, pr.nome_produto AS produto, pr.descontinuado,
           cat.nome_categoria AS categoria, ip.preco_unitario, ip.quantidade, ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)

conexao.close()

print("A camada Bronze tem", len(bronze.columns), "colunas:\n")
for posicao, coluna in enumerate(bronze.columns.tolist(), start=1):
    print(f"{posicao:>2}. {coluna}")

origem = {
    "id_pedido": "pedidos", 
    "data_pedido": "pedidos", 
    "data_prometida": "pedidos",
    "data_envio": "pedidos", 
    "frete": "pedidos", 
    "pais_entrega": "pedidos",
    "regiao_entrega": "pedidos", 
    "id_cliente": "clientes", 
    "cliente": "clientes",
    "cidade_cliente": "clientes", 
    "pais_cliente": "clientes", 
    "produto": "produtos",
    "descontinuado": "produtos", 
    "categoria": "categorias",
    "preco_unitario": "itens_pedido", 
    "quantidade": "itens_pedido", 
    "desconto": "itens_pedido",
}

mapa_colunas = pd.DataFrame({
    "coluna": bronze.columns,
    "tipo": [str(tipo) for tipo in bronze.dtypes],
    "veio_da_tabela": [origem[coluna] for coluna in bronze.columns],
})

print("\nDe onde veio cada coluna:")
display(mapa_colunas)

A camada Bronze tem 17 colunas:

 1. id_pedido
 2. data_pedido
 3. data_prometida
 4. data_envio
 5. frete
 6. pais_entrega
 7. regiao_entrega
 8. id_cliente
 9. cliente
10. cidade_cliente
11. pais_cliente
12. produto
13. descontinuado
14. categoria
15. preco_unitario
16. quantidade
17. desconto

De onde veio cada coluna:


C:\Users\laisa\AppData\Local\Temp\ipykernel_19676\1055287374.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  bronze = pd.read_sql('''


,coluna,tipo,veio_da_tabela
0,id_pedido,int64,pedidos
1,data_pedido,object,pedidos
2,data_prometida,object,pedidos
3,data_envio,object,pedidos
4,frete,float64,pedidos
5,pais_entrega,str,pedidos
6,regiao_entrega,str,pedidos
7,id_cliente,str,clientes
8,cliente,str,clientes
9,cidade_cliente,str,clientes


Essa tabela de origem é mais útil do que parece. Quando alguém perguntar "de onde saiu esse campo `categoria`?", a resposta não depende de memória: ela está escrita. Num ETL de verdade, esse mapa se chama **linhagem do dado** (ou *data lineage*), e existe exatamente para responder essa pergunta meses depois, quando ninguém lembra mais.

### 📖 Antes do código — provando o grão

A próxima célula usa `nunique()`, um método do Pandas que conta quantos **valores diferentes** existem numa coluna (a sigla vem de *number of unique*). Diferente do `len()`, que conta linhas, o `nunique()` conta valores distintos — e é justamente essa diferença que expõe o grão da tabela.

Se a coluna `id_pedido` tem 2155 linhas mas apenas 830 valores diferentes, isso significa que o mesmo pedido aparece repetido em várias linhas — uma para cada produto que ele contém.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    port="5433",
    dbname="northwind",
    user="postgres",
    password="admin",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, cl.nome_empresa AS cliente, pr.nome_produto AS produto,
           ip.preco_unitario, ip.quantidade
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
''', conexao)

print("Linhas no DataFrame:      ", len(bronze))
print("Pedidos diferentes:       ", bronze["id_pedido"].nunique())
print("Clientes diferentes:      ", bronze["cliente"].nunique())
print("Produtos diferentes:      ", bronze["produto"].nunique())

print("\nUm pedido que aparece em várias linhas — o pedido 10248:")
display(bronze[bronze["id_pedido"] == 10248])

display(bronze.iloc[[1000]])

conexao.close()

Linhas no DataFrame:       2155
Pedidos diferentes:        830
Clientes diferentes:       89
Produtos diferentes:       77

Um pedido que aparece em várias linhas — o pedido 10248:


C:\Users\laisa\AppData\Local\Temp\ipykernel_19676\1893989286.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  bronze = pd.read_sql('''


,id_pedido,cliente,produto,preco_unitario,quantidade
0,10248,Vins et alcools Chevalier,Queso Cabrales,14.0,12
1,10248,Vins et alcools Chevalier,Singaporean Hokkien Fried Mee,9.8,10
2,10248,Vins et alcools Chevalier,Mozzarella di Giovanni,34.8,5


,id_pedido,cliente,produto,preco_unitario,quantidade
1000,10626,Berglunds snabbköp,Perth Pasties,32.8,12


O pedido 10248 aparece em **3 linhas**, uma para cada produto que o cliente comprou naquela compra. O `id_pedido` se repete, o cliente se repete, e só o produto muda.

Agora a consequência prática, que é onde o erro nasce: se alguém somar a coluna `quantidade` e disser "a Northwind vendeu X unidades", está certo. Se contar as linhas e disser "a Northwind teve 2155 pedidos", está errado — foram 830. A mesma tabela responde as duas perguntas, mas cada pergunta exige uma contagem diferente.

É por isso que o grão precisa ser uma decisão consciente do ETL, escrita e conhecida por quem vai usar o dado depois — e não algo que cada analista descobre sozinho, do jeito difícil.

### ✏️ Atividade Prática 2 — Sua vez de programar

**Contextualização:** o time de compras da Northwind quer saber com quais fornecedores a empresa realmente trabalha hoje, para renegociar contratos. A tabela `suppliers` (uma das que ficaram com o nome em inglês) guarda esses fornecedores, e a tabela `produtos` liga cada produto ao seu fornecedor pela coluna `supplier_id`.

**Comando:** conecte no banco `northwind` e escreva uma consulta que junte `produtos` com `suppliers` (`JOIN suppliers s ON s.supplier_id = pr.supplier_id`), trazendo o nome do produto (`pr.nome_produto`), o nome da empresa fornecedora (`s.company_name`) e o país do fornecedor (`s.country`). Traga o resultado com `pd.read_sql()`, mostre quantas linhas vieram com `len()`, exiba as 10 primeiras com `display(resultado.head(10))` e feche a conexão.

In [ ]:
# escreva seu código aqui

import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    port="5433",
    dbname="northwind",
    user="postgres",
    password="admin",
)

fornecedores = pd.read_sql(
                            """
                            SELECT 
                                pr.nome_produto as produto,
                                s.company_name as fornecedor,
                                s.country as pais_fornecedor
                            FROM produtos pr
                                JOIN suppliers s 
                                ON s.supplier_id = pr.supplier_id
                            ORDER BY s.company_name, pr.nome_produto                            
                            """, conexao
                        )

display(f"O total de registros retornados é: {len(fornecedores)} linhas.")
display(fornecedores.head(10))

conexao.close()

C:\Users\laisa\AppData\Local\Temp\ipykernel_15344\1516097267.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fornecedores = pd.read_sql(


'O total de registros retornados é: 77 linhas.'

,produto,fornecedor,pais_fornecedor
0,Chartreuse verte,Aux joyeux ecclésiastiques,France
1,Côte de Blaye,Aux joyeux ecclésiastiques,France
2,Laughing Lumberjack Lager,Bigfoot Breweries,USA
3,Sasquatch Ale,Bigfoot Breweries,USA
4,Steeleye Stout,Bigfoot Breweries,USA
5,Queso Cabrales,Cooperativa de Quesos 'Las Cabras',Spain
6,Queso Manchego La Pastora,Cooperativa de Quesos 'Las Cabras',Spain
7,Escargots de Bourgogne,Escargots Nouveaux,France
8,Aniseed Syrup,Exotic Liquids,UK
9,Chang,Exotic Liquids,UK


## Passo 3 — Inspeção: conhecendo o dado antes de confiar nele

O dado está dentro do Python. Antes de fazer qualquer coisa com ele, você precisa responder quatro perguntas — e existe um método do Pandas para cada uma:

| Pergunta | Método | O que ele faz |
|---|---|---|
| Com o que eu estou lidando? | `head()` | mostra as primeiras linhas (5, por padrão) |
| Que tipo tem cada coluna? | `info()` | lista as colunas, o tipo de cada uma e quantos valores preenchidos tem |
| Como os números se comportam? | `describe()` | calcula contagem, média, desvio, mínimo, máximo e quartis das colunas numéricas |
| O que está faltando? | `isna().sum()` | conta quantos valores vazios existem em cada coluna |

Esses quatro você já conhece desde a Semana 06, quando aprendeu limpeza de dados. O que muda agora é o papel deles: lá, a inspeção servia para você entender um CSV; aqui, ela é uma **etapa formal do pipeline**. Um ETL profissional inspeciona antes de transformar, sempre — porque uma transformação escrita sem olhar o dado é um chute com cara de código.

### 📖 Antes do código — `shape`, `head()` e `info()`

`shape` é um atributo (não um método, por isso não leva parênteses) que devolve uma dupla com o número de linhas e de colunas: `(2155, 17)` significa 2155 linhas e 17 colunas.

`head(n)` devolve as `n` primeiras linhas do DataFrame — útil para ver a cara do dado sem despejar 2155 linhas na tela.

`info()` imprime um resumo técnico: o nome de cada coluna, quantos valores **não nulos** ela tem, e o tipo de dado (`int64` para inteiros, `float64` para decimais, `object` para texto, `datetime64` para datas). A coluna "Non-Null Count" é a mais reveladora: se uma coluna mostra menos valores não nulos do que o total de linhas, a diferença é exatamente a quantidade de buracos ali.

In [11]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    port="5433",
    dbname="northwind",
    user="postgres",
    password="admin",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, p.data_pedido, p.data_envio, p.ship_region AS regiao_entrega,
           cl.nome_empresa AS cliente, cl.pais AS pais_cliente,
           pr.nome_produto AS produto, pr.descontinuado, cat.nome_categoria AS categoria,
           ip.preco_unitario, ip.quantidade, ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)

print("Formato (linhas, colunas):", bronze.shape)
print("\nPrimeiras 5 linhas:")
display(bronze.head())

print("\nResumo técnico das colunas:")
bronze.info()

conexao.close()

Formato (linhas, colunas): (2155, 12)

Primeiras 5 linhas:


C:\Users\laisa\AppData\Local\Temp\ipykernel_15344\1390964077.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  bronze = pd.read_sql('''


,id_pedido,data_pedido,data_envio,regiao_entrega,cliente,pais_cliente,produto,descontinuado,categoria,preco_unitario,quantidade,desconto
0,10248,1996-07-04,1996-07-16,NaN,Vins et alcools Chevalier,France,Queso Cabrales,0,Dairy Products,14.0,12,0.0
1,10248,1996-07-04,1996-07-16,NaN,Vins et alcools Chevalier,France,Singaporean Hokkien Fried Mee,1,Grains/Cereals,9.8,10,0.0
2,10248,1996-07-04,1996-07-16,NaN,Vins et alcools Chevalier,France,Mozzarella di Giovanni,0,Dairy Products,34.8,5,0.0
3,10249,1996-07-05,1996-07-10,NaN,Toms Spezialitäten,Germany,Tofu,0,Produce,18.6,9,0.0
4,10249,1996-07-05,1996-07-10,NaN,Toms Spezialitäten,Germany,Manjimup Dried Apples,0,Produce,42.4,40,0.0



Resumo técnico das colunas:
<class 'pandas.DataFrame'>
RangeIndex: 2155 entries, 0 to 2154
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_pedido       2155 non-null   int64  
 1   data_pedido     2155 non-null   object 
 2   data_envio      2082 non-null   object 
 3   regiao_entrega  856 non-null    str    
 4   cliente         2155 non-null   str    
 5   pais_cliente    2155 non-null   str    
 6   produto         2155 non-null   str    
 7   descontinuado   2155 non-null   int64  
 8   categoria       2155 non-null   str    
 9   preco_unitario  2155 non-null   float64
 10  quantidade      2155 non-null   int64  
 11  desconto        2155 non-null   float64
dtypes: float64(2), int64(3), object(2), str(5)
memory usage: 202.2+ KB


Duas informações do `info()` merecem atenção agora, porque viram trabalho no Dia 2.

A primeira: `data_pedido` e `data_envio` aparecem como `object`, não como `datetime64`. `object` é como o Pandas chama "texto ou coisa genérica". Ou seja, para o Pandas, essas datas ainda **não são datas** — são objetos soltos. Enquanto estiverem assim, você não consegue calcular quantos dias se passaram entre uma e outra, nem agrupar vendas por mês.

A segunda: a coluna `data_envio` mostra menos valores não nulos do que as demais. Existem buracos ali, e cada buraco tem um significado de negócio que você vai precisar interpretar.

### 📖 Antes do código — `describe()`, `isna()` e `duplicated()`

`describe()` calcula estatísticas das colunas numéricas de uma vez: `count` (quantos valores), `mean` (média), `std` (desvio padrão), `min`, os quartis (`25%`, `50%`, `75%`) e `max`. Serve para farejar valor estranho — um preço mínimo negativo ou um máximo absurdo aparecem aqui antes de estragar a análise.

`isna()` devolve um DataFrame do mesmo tamanho, só que preenchido com `True` onde o valor está vazio e `False` onde está preenchido. Encadeando `.sum()`, o Pandas soma esses `True` (que valem 1) coluna por coluna — resultando na contagem de vazios de cada coluna.

`duplicated()` devolve uma coluna de `True`/`False` indicando quais linhas são cópias idênticas de alguma linha anterior; com `.sum()`, você tem o total de linhas repetidas.

In [12]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    port="5433",
    dbname="northwind",
    user="postgres",
    password="admin",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, p.data_pedido, p.data_envio, p.ship_region AS regiao_entrega,
           cl.nome_empresa AS cliente, cl.pais AS pais_cliente,
           pr.nome_produto AS produto, pr.descontinuado, cat.nome_categoria AS categoria,
           ip.preco_unitario, ip.quantidade, ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)

print("Estatísticas das colunas numéricas:")
display(bronze.describe())

print("\nValores vazios por coluna:")
display(bronze.isna().sum().to_frame("vazios"))

print("\nLinhas totalmente duplicadas:", bronze.duplicated().sum())

conexao.close()

Estatísticas das colunas numéricas:


C:\Users\laisa\AppData\Local\Temp\ipykernel_15344\3148900211.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  bronze = pd.read_sql('''


,id_pedido,descontinuado,preco_unitario,quantidade,desconto
count,2155.000000,2155.000000,2155.000000,2155.000000,2155.000000
mean,10659.375870,0.143852,26.218520,23.812993,0.056167
std,241.378032,0.351021,29.827418,19.022047,0.083450
min,10248.000000,0.000000,2.000000,1.000000,0.000000
25%,10451.000000,0.000000,12.000000,10.000000,0.000000
50%,10657.000000,0.000000,18.400000,20.000000,0.000000
75%,10862.500000,0.000000,32.000000,30.000000,0.100000
max,11077.000000,1.000000,263.500000,130.000000,0.250000



Valores vazios por coluna:


,vazios
id_pedido,0
data_pedido,0
data_envio,73
regiao_entrega,1299
cliente,0
pais_cliente,0
produto,0
descontinuado,0
categoria,0
preco_unitario,0



Linhas totalmente duplicadas: 0


O diagnóstico da camada Bronze, em números reais:

- **`data_envio` tem 73 linhas vazias.** Esses são itens de pedidos que ainda não saíram para entrega. Não é erro de digitação nem falha do sistema — é um pedido em aberto no momento em que o banco foi fotografado.
- **`regiao_entrega` tem 1299 linhas vazias**, mais da metade do total. Aqui o motivo é outro: muitos países simplesmente não usam o conceito de "região/estado" no endereço, então o campo nunca foi preenchido.
- **Nenhuma linha duplicada.** As chaves primárias do banco fizeram o trabalho delas.
- **Em `descontinuado`, o `describe()` mostra mínimo 0 e máximo 1.** É uma coluna de sim/não disfarçada de número: `0` para produto ativo, `1` para produto que a Northwind não vende mais.

Repare que os dois campos vazios pedem decisões **diferentes**, e é isso que torna o tratamento de nulos uma decisão de negócio, não uma regra técnica: um campo vazio pode significar "ainda não aconteceu" (`data_envio`) ou "não se aplica" (`regiao_entrega`). Apagar as duas situações com a mesma regra destruiria informação. No Dia 2 você decide cada uma separadamente.

### ✏️ Atividade Prática 3 — Sua vez de programar

**Contextualização:** a equipe de catálogo da Northwind desconfia que há produtos parados no estoque — itens que a empresa já descontinuou, mas cujas unidades continuam ocupando espaço no depósito. Antes de pedir um relatório completo, a gerente quer só um retrato rápido da situação do catálogo.

**Comando:** conecte no banco `northwind` e traga a tabela `produtos` inteira com `pd.read_sql('SELECT * FROM produtos', conexao)`. Depois, aplique a inspeção que você acabou de aprender, nesta ordem: mostre o formato com `.shape`, o resumo técnico com `.info()`, os vazios por coluna com `.isna().sum()`, e por fim conte quantos produtos estão descontinuados usando `(produtos["descontinuado"] == 1).sum()`. Feche a conexão ao final.

In [16]:
# escreva seu código aqui

import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    port="5433",
    dbname="northwind",
    user="postgres",
    password="admin",
)

produtos = pd.read_sql(
                        """
                        SELECT * FROM produtos
                        """, conexao
                    )

display(f"Linhas e colunas: {produtos.shape}")
produtos.info()

print("\nValores vazios por coluna:")
display(produtos.isna().sum().to_frame("vazios"))

descontinuados = (produtos["descontinuado"] == 1).sum()
print(f"Produtos descontinuados: {descontinuados} de {len(produtos)}.")


C:\Users\laisa\AppData\Local\Temp\ipykernel_15344\2535581336.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  produtos = pd.read_sql(


'Linhas e colunas: (77, 10)'

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_produto         77 non-null     int64  
 1   nome_produto       77 non-null     str    
 2   supplier_id        77 non-null     int64  
 3   id_categoria       77 non-null     int64  
 4   quantity_per_unit  77 non-null     str    
 5   unit_price         77 non-null     float64
 6   units_in_stock     77 non-null     int64  
 7   units_on_order     77 non-null     int64  
 8   reorder_level      77 non-null     int64  
 9   descontinuado      77 non-null     int64  
dtypes: float64(1), int64(7), str(2)
memory usage: 6.1 KB

Valores vazios por coluna:


,vazios
id_produto,0
nome_produto,0
supplier_id,0
id_categoria,0
quantity_per_unit,0
unit_price,0
units_in_stock,0
units_on_order,0
reorder_level,0
descontinuado,0


Produtos descontinuados: 10 de 77.


## Passo 4 — Materializando a camada Bronze

Até agora a extração vive só na memória do Python: quando você fechar o VS Code, ela some. **Materializar** uma camada significa gravá-la em algum lugar que sobrevive ao fim do programa — e é isso que transforma um script solto num pipeline de verdade.

A camada Bronze vai virar um arquivo `.csv` dentro de uma pasta própria. A estrutura que você vai construir ao longo desta semana é esta:

```
etl_northwind/
├── bronze/
│   ├── vendas_bruto.csv
│   └── metadados_extracao.csv
├── prata/          (Dia 2)
└── ouro/           (Dia 3)
```

### 📖 Antes do código — `makedirs()`, `to_csv()` e a confirmação

`os.makedirs(caminho, exist_ok=True)` cria a pasta indicada, inclusive as pastas intermediárias que faltarem no caminho. O parâmetro `exist_ok=True` diz "se a pasta já existir, tudo bem, siga em frente" — sem ele, rodar a célula uma segunda vez quebraria com um erro de pasta já existente.

`to_csv(caminho, index=False)` grava o DataFrame como arquivo de texto separado por vírgulas. O `index=False` evita que o Pandas escreva no arquivo aquela coluna de numeração (0, 1, 2, 3...) que ele usa internamente — essa numeração não é dado da Northwind, e incluí-la só sujaria o arquivo.

Depois de gravar, o código **reabre o próprio arquivo** com `pd.read_csv()` e mostra o conteúdo. Isso não é redundância: é a confirmação de que o arquivo existe mesmo e está legível. Uma mensagem "salvo com sucesso" impressa pelo seu próprio código não prova nada — quem prova é o arquivo reaberto do disco.

In [ ]:
import os
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido,
           p.data_pedido,
           p.required_date AS data_prometida,
           p.data_envio,
           p.freight AS frete,
           p.ship_country AS pais_entrega,
           p.ship_region AS regiao_entrega,
           cl.id_cliente,
           cl.nome_empresa AS cliente,
           cl.cidade AS cidade_cliente,
           cl.pais AS pais_cliente,
           pr.nome_produto AS produto,
           pr.descontinuado,
           cat.nome_categoria AS categoria,
           ip.preco_unitario,
           ip.quantidade,
           ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)

conexao.close()

os.makedirs("etl_northwind/bronze", exist_ok=True)
bronze.to_csv("etl_northwind/bronze/vendas_bruto.csv", index=False)

print("Camada Bronze gravada. Confirmando pela releitura do arquivo:")
confirmacao = pd.read_csv("etl_northwind/bronze/vendas_bruto.csv")
print("Linhas lidas de volta do arquivo:", len(confirmacao))
display(confirmacao.head())

### 👀 Veja com os próprios olhos, no VS Code

Rodar o código e ver a saída no notebook é metade da história. A outra metade é ver o arquivo existindo de verdade no seu computador — e isso vale a pena fazer agora, antes de seguir:

1. Clique no **ícone de pastas** (Explorer) na barra lateral esquerda do VS Code, ou pressione **Ctrl + Shift + E**.
2. Localize a pasta `etl_northwind`, que apareceu na mesma pasta onde este notebook está salvo.
3. Expanda `etl_northwind` → `bronze`, e **clique duas vezes** em `vendas_bruto.csv`.
4. O VS Code abre o arquivo numa aba nova. Você vai ver a primeira linha com os nomes das colunas separados por vírgula, e abaixo as 2155 linhas de dado.

Esse é o arquivo real da sua camada Bronze. Qualquer pessoa com esse `.csv` consegue refazer a sua análise sem tocar no banco de produção — que é exatamente o ponto da camada existir.

### 📖 Antes do código — vendo a estrutura de pastas pelo código

Além de olhar no Explorer, dá para o próprio Python listar o que foi criado, e isso é útil quando o pipeline cresce e as pastas se multiplicam.

`os.walk(caminho)` percorre uma pasta e todas as subpastas dela, devolvendo a cada volta três coisas: a pasta atual, a lista de subpastas dentro dela e a lista de arquivos dentro dela.

`os.path.getsize(caminho)` devolve o tamanho do arquivo em bytes — dividindo por 1024 você tem o tamanho em KB, que é mais fácil de ler.

`os.path.join(pasta, arquivo)` monta o caminho completo juntando pasta e nome de arquivo com a barra correta do sistema operacional. No Windows, usar `os.path.join` em vez de escrever `pasta + "/" + arquivo` na mão evita problemas com a barra invertida.

`sorted(lista)` devolve os itens da lista em ordem alfabética, sem alterar a lista original. Aqui ele garante que os arquivos apareçam sempre na mesma ordem, em vez da ordem em que o sistema operacional resolveu devolvê-los.

In [ ]:
import os

print("Estrutura criada até agora:\n")

for pasta_atual, subpastas, arquivos in os.walk("etl_northwind"):
    nivel = pasta_atual.replace("etl_northwind", "").count(os.sep)
    recuo = "    " * nivel
    print(f"{recuo}{os.path.basename(pasta_atual) or 'etl_northwind'}/")
    for arquivo in sorted(arquivos):
        caminho = os.path.join(pasta_atual, arquivo)
        tamanho_kb = os.path.getsize(caminho) / 1024
        print(f"{recuo}    {arquivo}  ({tamanho_kb:.1f} KB)")

### 📖 Antes do código — metadados: a certidão de nascimento da extração

Só o arquivo de dados não basta. Daqui a duas semanas, olhando esse CSV, você não vai saber responder: quando ele foi extraído? De qual banco? Quantas linhas tinha na origem? Essas informações **sobre** o dado se chamam **metadados**, e todo pipeline sério grava isso junto.

Três novidades no código abaixo:

`datetime.now()` devolve a data e a hora exatas deste instante. É assim que a extração fica carimbada no tempo.

`strftime("%d/%m/%Y %H:%M:%S")` converte esse instante em texto no formato brasileiro (dia/mês/ano hora:minuto:segundo). A sigla vem de *string format time*.

`pd.DataFrame([{...}])` cria um DataFrame a partir de uma lista de dicionários — cada dicionário vira uma linha, e cada chave vira uma coluna. Aqui a lista tem um dicionário só, porque os metadados desta extração cabem numa linha.

In [ ]:
import os
from datetime import datetime

import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, p.data_pedido, p.data_envio,
           cl.nome_empresa AS cliente, pr.nome_produto AS produto,
           ip.preco_unitario, ip.quantidade, ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
''', conexao)

conexao.close()

metadados = pd.DataFrame([{
    "camada": "bronze",
    "origem": "northwind (PostgreSQL local)",
    "tabela_base": "itens_pedido + pedidos + clientes + produtos",
    "linhas_extraidas": len(bronze),
    "colunas_extraidas": len(bronze.columns),
    "pedidos_distintos": bronze["id_pedido"].nunique(),
    "extraido_em": datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
}])

os.makedirs("etl_northwind/bronze", exist_ok=True)
metadados.to_csv("etl_northwind/bronze/metadados_extracao.csv", index=False)

print("Metadados gravados. Confirmando pela releitura do arquivo:")
display(pd.read_csv("etl_northwind/bronze/metadados_extracao.csv"))

### ✏️ Atividade Prática 4 — Sua vez de programar

**Contextualização:** o time de logística da Northwind vai analisar o custo de frete por transportadora, e pediu ao time de dados uma extração separada só com essa informação. Como é uma análise nova, ela precisa da própria camada Bronze — a extração de vendas que você acabou de fazer não traz o nome da transportadora.

**Comando:** conecte no banco `northwind` e extraia, com `pd.read_sql()`, uma tabela que junte `pedidos` com `shippers` (`JOIN shippers s ON s.shipper_id = p.ship_via`), trazendo `p.id_pedido`, `p.data_pedido`, `p.freight AS frete` e `s.company_name AS transportadora`. Grave o resultado em `etl_northwind/bronze/fretes_bruto.csv` com `to_csv(..., index=False)`, lembrando de criar a pasta com `os.makedirs(..., exist_ok=True)` antes. Confirme reabrindo o arquivo com `pd.read_csv()` e mostrando as primeiras linhas com `display()`. Por fim, abra o arquivo novo no Explorer do VS Code para ver o resultado com os próprios olhos. Feche a conexão.

In [ ]:
# escreva seu código aqui

### ✅ O que você fez no Dia 1

- Conectou o Python ao PostgreSQL com `psycopg2` e testou a conexão com `SELECT version()`.
- Viu o erro real de senha incorreta (`password authentication failed`) e aprendeu a reconhecê-lo.
- Consultou o catálogo do banco (`information_schema.tables`) para saber o que existe na origem **antes** de extrair.
- Descobriu o conceito de **grão**: 2155 itens distribuídos em 830 pedidos, e por que confundir os dois produz um número errado.
- Extraiu os dados de venda com um `JOIN` de cinco tabelas, trazendo o resultado direto para um DataFrame com `pd.read_sql()`.
- Documentou a **linhagem** das colunas: nome, tipo e tabela de origem de cada uma das 17 colunas.
- Inspecionou a extração com `shape`, `head()`, `info()`, `describe()`, `isna().sum()` e `duplicated()`.
- Diagnosticou os problemas reais do dado: 73 datas de envio vazias, 1299 regiões vazias, datas ainda em formato de texto e uma coluna de sim/não disfarçada de número.
- Materializou a camada **Bronze** em `etl_northwind/bronze/`, com o arquivo de dados e o arquivo de metadados — confirmou cada gravação reabrindo o arquivo do disco, listou a estrutura com `os.walk()` e abriu o `.csv` no Explorer do VS Code.

**No Dia 2** você pega essa camada Bronze e decide, uma por uma, o que fazer com cada problema que encontrou — construindo a camada Prata.

📚 **Para saber mais:** documentação oficial do Pandas em https://pandas.pydata.org/docs/ e do psycopg2 em https://www.psycopg.org/docs/

---

# 📅 DIA 2 — Transformação e camada Prata

No Dia 1 você extraiu 2155 linhas e descobriu quatro problemas concretos nelas. Neste encontro você resolve cada um — e o mais importante: **decide** o que fazer com cada um, em vez de aplicar uma receita pronta.

Um lembrete que vale para todo este encontro: a camada Bronze **não muda**. Ela continua lá, intacta, com os 2155 registros originais. Tudo que você fizer agora produz uma camada **nova**, a Prata. Se uma regra estiver errada, você corrige a regra e reconstrói a Prata a partir da Bronze — sem nunca precisar voltar ao banco de produção.

### 🔧 Antes de começar o Dia 2: garantindo as bibliotecas

Cada encontro começa com o VS Code recém-aberto, e o Python pode estar usando um ambiente diferente do da aula anterior (outro computador do laboratório, outro kernel escolhido). Por isso a instalação é repetida aqui. Se as bibliotecas já estiverem instaladas, o comando só confirma com a mensagem `Requirement already satisfied` ("requisito já atendido") e segue em frente — rodar de novo não faz mal nenhum.

Se, depois de instalar, uma célula reclamar com `ModuleNotFoundError: No module named ...`, clique em **Restart** no topo do notebook (reinicia o kernel) e rode a célula de novo. O kernel que já estava aberto antes da instalação não enxerga a biblioteca nova até ser reiniciado.

In [ ]:
%pip install psycopg2-binary pandas matplotlib

## Passo 5 — De problema a decisão

Esta é a tabela de decisões da camada Prata. Cada linha é um problema que você encontrou no Dia 1, o que ele significa no negócio da Northwind, e a decisão que essa interpretação justifica:

| Problema encontrado | O que significa no negócio | Decisão para a Prata |
|---|---|---|
| `data_pedido` e `data_envio` são texto, não data | O sistema entregou a data como texto simples | **Converter** para o tipo data do Pandas |
| 73 linhas sem `data_envio` | Pedidos ainda não despachados | **Remover** — venda não entregue não entra em análise de entrega |
| 1299 linhas sem `regiao_entrega` | Muitos países não usam região no endereço | **Preencher** com "NAO INFORMADO" — não é erro, é ausência legítima |
| 310 linhas de produtos descontinuados | Produtos que a Northwind não vende mais | **Remover** — o painel analisa o catálogo ativo |
| Não existe coluna de receita | O banco guarda preço, quantidade e desconto separados | **Criar** a coluna calculando o valor final |

Repare que os dois campos vazios tiveram destinos opostos: um foi removido, o outro preenchido. A diferença não é técnica — é o significado de cada ausência. Essa é a parte do ETL que nenhuma biblioteca resolve sozinha.

**Uma decisão importante sobre a ordem:** primeiro converter as datas, depois remover linhas, depois criar as colunas novas. A ordem importa porque cada passo depende do anterior — não dá para calcular "dias até a entrega" antes de a data virar data de verdade, nem faz sentido calcular receita de linhas que serão descartadas em seguida.

## Passo 6 — Transformação 1: datas de verdade

### ⚠️ Um erro real: subtrair datas que ainda são texto

Antes de converter, veja por que a conversão é necessária. A célula abaixo tenta calcular quantos dias se passaram entre o pedido e o envio — subtraindo uma coluna da outra, do jeito mais natural possível.

Só que essas colunas ainda chegaram do banco como `object`. O `try`/`except` captura o erro e mostra a mensagem; `TypeError` é o erro que o Python usa quando uma operação não faz sentido para aquele tipo de dado.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, p.data_pedido, p.required_date AS data_prometida, p.data_envio,
           p.freight AS frete, p.ship_country AS pais_entrega, p.ship_region AS regiao_entrega,
           cl.id_cliente, cl.nome_empresa AS cliente, cl.cidade AS cidade_cliente,
           cl.pais AS pais_cliente, pr.nome_produto AS produto, pr.descontinuado,
           cat.nome_categoria AS categoria, ip.preco_unitario, ip.quantidade, ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)
conexao.close()

print("Tipo da coluna data_pedido:", bronze["data_pedido"].dtype)
print("Tipo da coluna data_envio: ", bronze["data_envio"].dtype)

try:
    dias = bronze["data_envio"] - bronze["data_pedido"]
    print(dias.head())
except TypeError as erro:
    print("\nDeu erro, e é este aqui:")
    print(erro)

A mensagem `unsupported operand type(s) for -: 'str' and 'str'` diz exatamente o que aconteceu: o Python não sabe subtrair um texto de outro texto. Para ele, `"1996-07-16"` é uma sequência de caracteres como qualquer outra — não um ponto no tempo.

### 📖 Antes do código — `pd.to_datetime()`

`pd.to_datetime(coluna)` converte uma coluna de texto (ou de datas soltas) para o tipo `datetime64`, que é o tipo de data nativo do Pandas. Depois dessa conversão, três coisas passam a funcionar: subtrair uma data da outra (resultando numa duração), comparar datas com `>` e `<`, e extrair partes da data com o acessador `.dt` (`.dt.year`, `.dt.month`, `.dt.day`).

O código abaixo converte as três colunas de data de uma vez, percorrendo uma lista com `for`. Depois da conversão, ele mostra o tipo novo de cada coluna e refaz a subtração que acabou de falhar — agora com sucesso.

`.dt.days` extrai apenas o número de dias de uma duração. Sem ele, o resultado apareceria como `7 days`, um objeto de duração; com ele, você tem o número `7`, que dá para somar e tirar média.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, p.data_pedido, p.required_date AS data_prometida, p.data_envio,
           p.freight AS frete, p.ship_country AS pais_entrega, p.ship_region AS regiao_entrega,
           cl.id_cliente, cl.nome_empresa AS cliente, cl.cidade AS cidade_cliente,
           cl.pais AS pais_cliente, pr.nome_produto AS produto, pr.descontinuado,
           cat.nome_categoria AS categoria, ip.preco_unitario, ip.quantidade, ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)
conexao.close()

print("ANTES da conversão:")
print(bronze[["data_pedido", "data_prometida", "data_envio"]].dtypes.to_string())

for coluna in ["data_pedido", "data_prometida", "data_envio"]:
    bronze[coluna] = pd.to_datetime(bronze[coluna])

print("\nDEPOIS da conversão:")
print(bronze[["data_pedido", "data_prometida", "data_envio"]].dtypes.to_string())

print("\nAgora a subtração funciona:")
dias = (bronze["data_envio"] - bronze["data_pedido"]).dt.days
display(dias.head().to_frame("dias_ate_o_envio"))

### 📖 Antes do código — o que a data vira depois de convertida

Com o tipo certo, o acessador `.dt` abre um conjunto de informações que estavam escondidas dentro da data.

`.dt.year` e `.dt.month` extraem o ano e o mês como números inteiros. `.dt.day_name()` devolve o nome do dia da semana em texto. `.dt.to_period("M")` transforma a data no período mensal a que ela pertence (`1996-07-04` vira `1996-07`), que é exatamente o que se usa para agrupar vendas por mês.

`.astype(str)` converte uma coluna para texto. Ela aparece logo depois do `to_period("M")` porque o período é um tipo especial do Pandas, e transformá-lo em texto simples deixa o valor pronto para ser gravado em CSV, comparado ou usado como rótulo de gráfico.

`value_counts()` conta quantas vezes cada valor diferente aparece numa coluna — é como um `GROUP BY` do SQL resumido num método só. Encadeando `.sort_index()`, o resultado sai em ordem de rótulo em vez de ordem de frequência.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, p.data_pedido, p.required_date AS data_prometida, p.data_envio,
           p.freight AS frete, p.ship_country AS pais_entrega, p.ship_region AS regiao_entrega,
           cl.id_cliente, cl.nome_empresa AS cliente, cl.cidade AS cidade_cliente,
           cl.pais AS pais_cliente, pr.nome_produto AS produto, pr.descontinuado,
           cat.nome_categoria AS categoria, ip.preco_unitario, ip.quantidade, ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)
conexao.close()

for coluna in ["data_pedido", "data_prometida", "data_envio"]:
    bronze[coluna] = pd.to_datetime(bronze[coluna])

exemplo = pd.DataFrame({
    "data_pedido": bronze["data_pedido"],
    "ano": bronze["data_pedido"].dt.year,
    "mes": bronze["data_pedido"].dt.month,
    "dia_da_semana": bronze["data_pedido"].dt.day_name(),
    "periodo_mensal": bronze["data_pedido"].dt.to_period("M").astype(str),
})

print("A mesma data, decomposta:")
display(exemplo.head())

print("\nItens vendidos por ano:")
display(bronze["data_pedido"].dt.year.value_counts().sort_index().to_frame("itens_vendidos"))

### ✏️ Atividade Prática 5 — Sua vez de programar

**Contextualização:** a equipe de operações da Northwind quer entender se existe concentração de pedidos em algum dia da semana. Se a maioria das compras cai sempre na segunda-feira, a escala da equipe de separação precisa mudar — e hoje ninguém sabe responder isso.

**Comando:** conecte no banco `northwind`, extraia a tabela `pedidos` com `pd.read_sql('SELECT id_pedido, data_pedido FROM pedidos', conexao)` e feche a conexão. Converta a coluna `data_pedido` com `pd.to_datetime()`, crie uma coluna nova chamada `dia_da_semana` usando `.dt.day_name()`, e mostre a contagem de pedidos por dia da semana com `value_counts()`. Exiba o resultado com `display()`.

In [ ]:
# escreva seu código aqui

## Passo 7 — Transformação 2: decidindo o destino de cada vazio

Agora as duas decisões sobre valores ausentes — e elas vão em direções opostas, de propósito.

### 📖 Antes do código — `dropna()` com critério

`dropna()` remove linhas que tenham valor vazio. Sozinho, ele remove qualquer linha com vazio em **qualquer** coluna — e isso aqui seria um desastre: como `regiao_entrega` tem 1299 vazios, você perderia mais da metade do dado por causa de uma coluna que nem é essencial.

O parâmetro `subset=["data_envio"]` resolve isso: ele diz "olhe o vazio **apenas** nesta coluna". As demais colunas podem ter vazios à vontade, que a linha permanece.

O código também guarda a contagem antes e depois, e mostra a diferença. Registrar quantas linhas cada regra descartou não é capricho: é o que permite explicar, depois, por que o relatório tem menos vendas do que o sistema de origem.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, p.data_pedido, p.required_date AS data_prometida, p.data_envio,
           p.freight AS frete, p.ship_country AS pais_entrega, p.ship_region AS regiao_entrega,
           cl.id_cliente, cl.nome_empresa AS cliente, cl.cidade AS cidade_cliente,
           cl.pais AS pais_cliente, pr.nome_produto AS produto, pr.descontinuado,
           cat.nome_categoria AS categoria, ip.preco_unitario, ip.quantidade, ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)
conexao.close()

for coluna in ["data_pedido", "data_prometida", "data_envio"]:
    bronze[coluna] = pd.to_datetime(bronze[coluna])

linhas_antes = len(bronze)
prata = bronze.dropna(subset=["data_envio"])
linhas_depois = len(prata)

print("Linhas antes da regra:  ", linhas_antes)
print("Linhas depois da regra: ", linhas_depois)
print("Linhas descartadas:     ", linhas_antes - linhas_depois)
print("\nMotivo do descarte: pedido ainda não despachado (sem data de envio).")

print("\nTabela resultante depois da remoção:")
display(prata.head())

73 linhas descartadas, 2082 mantidas. Essas 73 não são lixo — são pedidos legítimos, que simplesmente ainda não saíram do depósito no dia em que o banco foi fotografado. Elas continuam vivas na camada Bronze, e se amanhã alguém quiser analisar "pedidos em aberto", é para lá que essa pessoa vai.

### 📖 Antes do código — `fillna()` e a ausência que é informação

Para `regiao_entrega` a decisão é oposta: preencher em vez de remover.

`fillna(valor)` substitui todo valor vazio de uma coluna pelo valor que você indicar. Aqui o valor escolhido é o texto `"NAO INFORMADO"` — e essa escolha é deliberada. Preencher com `0` ou com texto vazio faria o vazio desaparecer disfarçadamente; preencher com uma etiqueta explícita mantém visível que ali nunca houve informação, o que é diferente de um erro.

A linha `prata = prata.copy()` antes da alteração evita um aviso do Pandas (`SettingWithCopyWarning`). Esse aviso aparece quando você altera um DataFrame que foi criado a partir de um filtro de outro — o Pandas não consegue garantir se você quis alterar a cópia ou o original. O `.copy()` deixa explícito que é uma cópia independente, e o aviso some.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, p.data_pedido, p.required_date AS data_prometida, p.data_envio,
           p.freight AS frete, p.ship_country AS pais_entrega, p.ship_region AS regiao_entrega,
           cl.id_cliente, cl.nome_empresa AS cliente, cl.cidade AS cidade_cliente,
           cl.pais AS pais_cliente, pr.nome_produto AS produto, pr.descontinuado,
           cat.nome_categoria AS categoria, ip.preco_unitario, ip.quantidade, ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)
conexao.close()

for coluna in ["data_pedido", "data_prometida", "data_envio"]:
    bronze[coluna] = pd.to_datetime(bronze[coluna])

prata = bronze.dropna(subset=["data_envio"]).copy()

print("Vazios em regiao_entrega ANTES:", prata["regiao_entrega"].isna().sum())
prata["regiao_entrega"] = prata["regiao_entrega"].fillna("NAO INFORMADO")
print("Vazios em regiao_entrega DEPOIS:", prata["regiao_entrega"].isna().sum())

print("\nComo a coluna ficou (valores mais frequentes):")
display(prata["regiao_entrega"].value_counts().head().to_frame("linhas"))

print("\nTabela resultante:")
display(prata.head())

### ✏️ Atividade Prática 6 — Sua vez de programar

**Contextualização:** o time de atendimento da Northwind quer montar uma campanha por telefone para os clientes cadastrados. O problema é que o cadastro tem campos incompletos: nem todo cliente informou fax, e nem todo cliente tem região preenchida. Antes de entregar a lista, o time de dados precisa deixar claro quais contatos estão realmente disponíveis, sem sumir com nenhum cliente da lista.

**Comando:** conecte no banco `northwind`, extraia a tabela `clientes` com `pd.read_sql('SELECT id_cliente, nome_empresa, nome_contato, phone, fax, region FROM clientes', conexao)` e feche a conexão. Conte os vazios de cada coluna com `.isna().sum()`. Depois preencha os vazios das colunas `fax` e `region` com o texto `"NAO INFORMADO"` usando `fillna()`, sem remover nenhuma linha. Confirme que sobraram zero vazios e mostre a tabela final com `display()`.

In [ ]:
# escreva seu código aqui

## Passo 8 — Transformação 3: regra de negócio e colunas novas

Faltam duas transformações: aplicar a regra que exclui produtos descontinuados, e criar as colunas que o banco não tem.

### 📖 Antes do código — filtro por regra de negócio

O filtro `prata[prata["descontinuado"] == 0]` seleciona apenas as linhas em que a coluna vale 0, ou seja, produtos ativos. A parte de dentro dos colchetes produz uma coluna de `True`/`False`, e o DataFrame devolve só as linhas marcadas como `True` — é a mesma seleção booleana que você usa desde a Semana 05.

Vale nomear a decisão que está por trás desse filtro: o painel que a diretoria pediu analisa o **catálogo ativo**, para decidir compra e reposição. Um produto descontinuado não pode ser reposto, então mantê-lo inflaria o número sem que ninguém pudesse agir sobre ele. Se amanhã a pergunta mudar para "quanto perdemos ao descontinuar produtos?", essa regra deixa de valer — e a resposta virá da camada Bronze, que preservou essas linhas.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, p.data_pedido, p.required_date AS data_prometida, p.data_envio,
           p.freight AS frete, p.ship_country AS pais_entrega, p.ship_region AS regiao_entrega,
           cl.id_cliente, cl.nome_empresa AS cliente, cl.cidade AS cidade_cliente,
           cl.pais AS pais_cliente, pr.nome_produto AS produto, pr.descontinuado,
           cat.nome_categoria AS categoria, ip.preco_unitario, ip.quantidade, ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)
conexao.close()

for coluna in ["data_pedido", "data_prometida", "data_envio"]:
    bronze[coluna] = pd.to_datetime(bronze[coluna])

prata = bronze.dropna(subset=["data_envio"]).copy()
prata["regiao_entrega"] = prata["regiao_entrega"].fillna("NAO INFORMADO")

antes = len(prata)
prata = prata[prata["descontinuado"] == 0].copy()
print("Linhas antes de remover descontinuados:", antes)
print("Linhas depois:                         ", len(prata))
print("Descartadas por produto descontinuado: ", antes - len(prata))

print("\nConferindo que sobrou só produto ativo:")
display(prata["descontinuado"].value_counts().to_frame("linhas"))

### ⚠️ Antes de criar a receita, um detalhe que engana muita gente

A receita de cada item é `preço × quantidade × (1 − desconto)`. Parece uma multiplicação trivial, mas o resultado sai assim:

```
1261.3999999999999
222.29999999999998
604.8000000000001
```

Isso não é bug do seu código: é como o computador guarda números com casas decimais. Internamente ele trabalha em base 2, e certos valores decimais (como 0,1) não têm representação exata nessa base — o resultado fica com um resíduo minúsculo. **308 das linhas desta extração** terminam com esse tipo de sobra.

Num relatório financeiro isso é inaceitável: ninguém aceita "R$ 1261,3999999999999". A solução é arredondar no momento de criar a coluna.

### 📖 Antes do código — criando colunas derivadas

`round(2)` arredonda para duas casas decimais — o padrão para valores em dinheiro.

As três colunas novas nascem de contas entre colunas que já existem:

| Coluna nova | Como é calculada | Para que serve |
|---|---|---|
| `receita` | `preco_unitario × quantidade × (1 − desconto)`, arredondado | o valor que efetivamente entrou |
| `dias_entrega` | `data_envio − data_pedido`, em dias | medir o tempo de atendimento |
| `entrega_atrasada` | `data_envio > data_prometida` | marcar se o prazo foi furado |

A coluna `entrega_atrasada` nasce de uma comparação, então ela é do tipo booleano (`True`/`False`). Isso tem uma vantagem prática: no Pandas, `True` conta como 1 e `False` como 0, então `.sum()` nessa coluna já devolve quantas entregas atrasaram, e `.mean()` devolve a proporção.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, p.data_pedido, p.required_date AS data_prometida, p.data_envio,
           p.freight AS frete, p.ship_country AS pais_entrega, p.ship_region AS regiao_entrega,
           cl.id_cliente, cl.nome_empresa AS cliente, cl.cidade AS cidade_cliente,
           cl.pais AS pais_cliente, pr.nome_produto AS produto, pr.descontinuado,
           cat.nome_categoria AS categoria, ip.preco_unitario, ip.quantidade, ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)
conexao.close()

for coluna in ["data_pedido", "data_prometida", "data_envio"]:
    bronze[coluna] = pd.to_datetime(bronze[coluna])

prata = bronze.dropna(subset=["data_envio"]).copy()
prata["regiao_entrega"] = prata["regiao_entrega"].fillna("NAO INFORMADO")
prata = prata[prata["descontinuado"] == 0].copy()

prata["receita"] = (prata["preco_unitario"] * prata["quantidade"] * (1 - prata["desconto"])).round(2)
prata["dias_entrega"] = (prata["data_envio"] - prata["data_pedido"]).dt.days
prata["entrega_atrasada"] = prata["data_envio"] > prata["data_prometida"]

print("Receita total da camada Prata: R$", round(prata["receita"].sum(), 2))
print("Tempo médio de entrega:", round(prata["dias_entrega"].mean(), 1), "dias")
print("Entrega mais rápida:", prata["dias_entrega"].min(), "dia(s) | mais lenta:", prata["dias_entrega"].max(), "dias")
print("Entregas atrasadas:", int(prata["entrega_atrasada"].sum()), "de", len(prata))

print("\nAs três colunas novas, ao lado das que as originaram:")
display(prata[["produto", "preco_unitario", "quantidade", "desconto", "receita",
               "data_pedido", "data_envio", "dias_entrega", "entrega_atrasada"]].head())

### ✏️ Atividade Prática 7 — Sua vez de programar

**Contextualização:** a diretoria da Northwind quer saber quanto o frete pesa em relação ao valor dos pedidos. A suspeita é que, em alguns pedidos pequenos, o frete chega perto do valor da própria mercadoria — e isso mudaria a política de pedido mínimo.

**Comando:** conecte no banco `northwind`, extraia com `pd.read_sql()` uma consulta que junte `itens_pedido` e `pedidos` trazendo `p.id_pedido`, `p.freight AS frete`, `ip.preco_unitario`, `ip.quantidade` e `ip.desconto`, e feche a conexão. Crie a coluna `receita` com a mesma fórmula do exemplo acima, lembrando do `.round(2)`. Depois crie a coluna `frete_sobre_receita`, dividindo `frete` por `receita` e multiplicando por 100, também arredondada para 2 casas. Por fim, mostre com `display()` as 10 linhas com maior `frete_sobre_receita`, usando `.sort_values("frete_sobre_receita", ascending=False).head(10)`.

In [ ]:
# escreva seu código aqui

## Passo 9 — Materializando a camada Prata

As cinco transformações estão decididas e testadas. Agora elas viram um bloco único — o código que constrói a camada Prata do começo ao fim — e o resultado é gravado em disco.

### 📖 Antes do código — o pipeline da Prata, e o relatório de transformação

O código abaixo não traz nenhuma função nova: ele junta, na ordem certa, tudo que você fez nos Passos 6, 7 e 8. O que ele acrescenta é o **relatório de transformação**: um DataFrame que registra quantas linhas existiam em cada etapa do caminho.

Esse relatório é o que responde a pergunta que sempre aparece na reunião: "por que o painel mostra menos vendas do que o sistema?". Com ele, a resposta é uma tabela, não uma suposição.

In [ ]:
import os
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

bronze = pd.read_sql('''
    SELECT p.id_pedido, p.data_pedido, p.required_date AS data_prometida, p.data_envio,
           p.freight AS frete, p.ship_country AS pais_entrega, p.ship_region AS regiao_entrega,
           cl.id_cliente, cl.nome_empresa AS cliente, cl.cidade AS cidade_cliente,
           cl.pais AS pais_cliente, pr.nome_produto AS produto, pr.descontinuado,
           cat.nome_categoria AS categoria, ip.preco_unitario, ip.quantidade, ip.desconto
    FROM itens_pedido ip
    JOIN pedidos p ON p.id_pedido = ip.id_pedido
    JOIN clientes cl ON cl.id_cliente = p.id_cliente
    JOIN produtos pr ON pr.id_produto = ip.id_produto
    JOIN categorias cat ON cat.id_categoria = pr.id_categoria
''', conexao)
conexao.close()

etapas = [{"etapa": "1. Extração bruta (Bronze)", "linhas": len(bronze)}]

for coluna in ["data_pedido", "data_prometida", "data_envio"]:
    bronze[coluna] = pd.to_datetime(bronze[coluna])
etapas.append({"etapa": "2. Datas convertidas", "linhas": len(bronze)})

prata = bronze.dropna(subset=["data_envio"]).copy()
etapas.append({"etapa": "3. Sem pedidos não despachados", "linhas": len(prata)})

prata["regiao_entrega"] = prata["regiao_entrega"].fillna("NAO INFORMADO")
etapas.append({"etapa": "4. Região vazia preenchida", "linhas": len(prata)})

prata = prata[prata["descontinuado"] == 0].copy()
etapas.append({"etapa": "5. Sem produtos descontinuados", "linhas": len(prata)})

prata["receita"] = (prata["preco_unitario"] * prata["quantidade"] * (1 - prata["desconto"])).round(2)
prata["dias_entrega"] = (prata["data_envio"] - prata["data_pedido"]).dt.days
prata["entrega_atrasada"] = prata["data_envio"] > prata["data_prometida"]
etapas.append({"etapa": "6. Colunas derivadas criadas", "linhas": len(prata)})

os.makedirs("etl_northwind/prata", exist_ok=True)
prata.to_csv("etl_northwind/prata/vendas_limpo.csv", index=False)

print("Relatório de transformação:")
display(pd.DataFrame(etapas))

print("\nCamada Prata gravada. Confirmando pela releitura do arquivo:")
confirmacao = pd.read_csv("etl_northwind/prata/vendas_limpo.csv")
print("Linhas lidas de volta:", len(confirmacao), "| Colunas:", len(confirmacao.columns))
display(confirmacao.head())

### 📖 Antes do código — comparando Bronze e Prata lado a lado

A camada Prata nasceu da Bronze, mas as duas são diferentes de propósito. Vale enxergar essa diferença em números, e é isso que o código abaixo faz.

`set(lista)` cria um conjunto a partir de uma lista. Conjuntos permitem a operação `-` (diferença), que devolve o que existe num conjunto e não existe no outro — é assim que o código descobre quais colunas a Prata ganhou em relação à Bronze.

`sorted()` devolve os elementos em ordem alfabética, para a leitura ficar previsível.

In [ ]:
import pandas as pd

bronze = pd.read_csv("etl_northwind/bronze/vendas_bruto.csv")
prata = pd.read_csv("etl_northwind/prata/vendas_limpo.csv")

comparacao = pd.DataFrame([
    {"camada": "bronze", "linhas": len(bronze), "colunas": len(bronze.columns),
     "arquivo": "etl_northwind/bronze/vendas_bruto.csv"},
    {"camada": "prata", "linhas": len(prata), "colunas": len(prata.columns),
     "arquivo": "etl_northwind/prata/vendas_limpo.csv"},
])
display(comparacao)

colunas_novas = sorted(set(prata.columns) - set(bronze.columns))
print("Colunas que a Prata ganhou:", colunas_novas)

print("\nTipo de cada coluna na Prata:")
display(prata.dtypes.to_frame("tipo"))

### 👀 Veja no VS Code: a pasta prata e as colunas novas

Vale repetir o hábito do Dia 1, porque agora tem coisa nova para ver:

1. Abra o Explorer (**Ctrl + Shift + E**) e expanda `etl_northwind`. Agora existem **duas** subpastas: `bronze/` e `prata/`.
2. Clique duas vezes em `prata/vendas_limpo.csv`.
3. Olhe a **primeira linha** do arquivo, que traz os nomes das colunas. Vá até o final dela: as três últimas são `receita`, `dias_entrega` e `entrega_atrasada` — as colunas que não existiam no banco e que o seu código criou.
4. Abra também `bronze/vendas_bruto.csv` numa aba ao lado e compare: a primeira linha da Bronze termina em `desconto`, e não tem nenhuma das três colunas novas.

Essa comparação lado a lado é o melhor resumo visual do que a camada Prata faz: mesmas vendas, menos linhas (porque algumas foram descartadas por regra) e mais colunas (porque outras foram calculadas).

A célula abaixo lista a estrutura completa pelo código, do mesmo jeito que no Dia 1 — agora com as duas camadas.

In [ ]:
import os

print("Estrutura do pipeline até agora:\n")

for pasta_atual, subpastas, arquivos in os.walk("etl_northwind"):
    nivel = pasta_atual.replace("etl_northwind", "").count(os.sep)
    recuo = "    " * nivel
    print(f"{recuo}{os.path.basename(pasta_atual) or 'etl_northwind'}/")
    for arquivo in sorted(arquivos):
        caminho = os.path.join(pasta_atual, arquivo)
        tamanho_kb = os.path.getsize(caminho) / 1024
        print(f"{recuo}    {arquivo}  ({tamanho_kb:.1f} KB)")

### ✅ O que você fez no Dia 2

- Transformou cada problema encontrado no Dia 1 numa **decisão documentada**, com a justificativa de negócio ao lado.
- Viu o erro real de subtrair datas que ainda eram texto (`unsupported operand type(s) for -: 'str' and 'str'`).
- Converteu as datas com `pd.to_datetime()` e passou a extrair ano, mês, dia da semana e período mensal com o acessador `.dt`.
- Removeu 73 linhas de pedidos não despachados com `dropna(subset=["data_envio"])` — e entendeu por que o `subset` é obrigatório aqui.
- Preencheu as regiões vazias com `fillna("NAO INFORMADO")`, tratando ausência legítima de forma diferente de dado faltante.
- Aplicou a regra de negócio que remove produtos descontinuados, reduzindo de 2082 para 1784 linhas.
- Descobriu a imprecisão de ponto flutuante em 308 linhas e resolveu com `.round(2)` na criação da coluna `receita`.
- Criou três colunas que não existiam no banco: `receita`, `dias_entrega` e `entrega_atrasada`.
- Materializou a camada **Prata**, gerou o **relatório de transformação** com o total de linhas em cada etapa, e comparou Bronze × Prata em linhas, colunas e tipos.

**No Dia 3** você agrega essa camada Prata na camada Ouro, transforma o resultado em gráficos e carrega tudo de volta no PostgreSQL — com validação e controle de falha.

📚 **Para saber mais:** a página sobre dados ausentes na documentação oficial do Pandas: https://pandas.pydata.org/docs/user_guide/missing_data.html

---

# 📅 DIA 3 — Camada Ouro, carga e validação

Você tem uma camada Prata com 1784 linhas limpas e 20 colunas. Neste último encontro você faz três coisas: **agrega** essa camada em indicadores (Ouro), **mostra** esses indicadores em gráficos, e **carrega** o resultado de volta no PostgreSQL — com validação, controle de falha e registro de execução.

A partir de agora o ponto de partida não é mais o banco: é o arquivo `prata/vendas_limpo.csv` que você gerou no Dia 2. Isso é proposital, e é assim que um pipeline real funciona — cada etapa consome a camada anterior, não a origem.

### 🔧 Antes de começar o Dia 3: garantindo as bibliotecas

Mesmo motivo do Dia 2: o VS Code foi aberto de novo e o ambiente pode não ser o mesmo. Neste encontro entram três recursos que dependem dessas instalações — o `matplotlib` para os gráficos, o módulo `psycopg2.extras` (que já vem dentro do `psycopg2-binary`) para a carga em lote, e o Pandas para a leitura das camadas.

Se aparecer `ModuleNotFoundError` depois de instalar, clique em **Restart** no topo do notebook e rode a célula outra vez.

In [ ]:
%pip install psycopg2-binary pandas matplotlib

## Passo 10 — Camada Ouro: de 1784 linhas para 8 números

A camada Ouro responde perguntas de negócio. Ninguém abre a camada Ouro para ver uma venda específica — abre para saber qual categoria vende mais, qual mês foi o melhor, quem são os maiores clientes.

Tecnicamente, isso é **agregação**: juntar muitas linhas numa só, aplicando uma conta. É o mesmo `GROUP BY` que você usou em SQL nas Semanas 09 e 10, agora no Pandas.

### 📖 Antes do código — `groupby()` e `agg()`

`groupby("coluna")` separa o DataFrame em grupos, um para cada valor diferente daquela coluna. Sozinho ele não calcula nada — só organiza as linhas em caixinhas.

`agg({...})` diz qual conta fazer dentro de cada caixinha. Na forma usada abaixo, cada linha do `agg()` cria uma coluna nova no resultado, no formato `nome_da_coluna_nova=("coluna_de_origem", "operação")`:

| Operação | O que faz |
|---|---|
| `"sum"` | soma os valores do grupo |
| `"mean"` | calcula a média do grupo |
| `"nunique"` | conta quantos valores **diferentes** existem no grupo |
| `"count"` | conta quantas linhas o grupo tem |

Repare no uso de `"nunique"` para contar pedidos: pelo que você aprendeu sobre grão no Dia 1, contar linhas daria o número de **itens**, não de pedidos. O `nunique` sobre `id_pedido` é o que devolve a contagem correta de pedidos.

`reset_index()` transforma o resultado do agrupamento de volta num DataFrame comum, com a coluna agrupada virando coluna normal em vez de índice. `sort_values("coluna", ascending=False)` ordena do maior para o menor.

In [ ]:
import pandas as pd

prata = pd.read_csv("etl_northwind/prata/vendas_limpo.csv")

ouro_categoria = (
    prata.groupby("categoria")
    .agg(
        receita_total=("receita", "sum"),
        itens_vendidos=("quantidade", "sum"),
        pedidos=("id_pedido", "nunique"),
        dias_entrega_medio=("dias_entrega", "mean"),
    )
    .round(2)
    .reset_index()
    .sort_values("receita_total", ascending=False)
)

print("A camada Prata tinha", len(prata), "linhas.")
print("A camada Ouro por categoria tem", len(ouro_categoria), "linhas.\n")
display(ouro_categoria)

print("Conferência: a soma das categorias bate com a receita total da Prata?")
print("Soma na Ouro: ", round(ouro_categoria["receita_total"].sum(), 2))
print("Soma na Prata:", round(prata["receita"].sum(), 2))

1784 linhas viraram 8. E a conferência no fim da célula não é enfeite: ela prova que a agregação não perdeu nem inventou dinheiro no caminho. Sempre que você agregar, confira se a soma do resultado bate com a soma da origem — é a forma mais barata de pegar um `groupby` escrito na coluna errada.

O resultado também já conta uma história de negócio: **Dairy Products** e **Beverages** disputam a liderança com cerca de R$ 230 mil cada, enquanto **Meat/Poultry** fica em R$ 21 mil — dez vezes menos. E a coluna `dias_entrega_medio` mostra que Produce é a categoria que sai mais rápido do depósito (7 dias), contra 9 de Beverages.

### 📖 Antes do código — agrupando por mais de uma coluna

`groupby()` aceita uma **lista** de colunas. Nesse caso, cada grupo passa a ser uma combinação de valores — aqui, cada cliente com seu identificador e país juntos, para o resultado já sair com o nome e a localização.

`head(10)` no fim, depois da ordenação, é o que transforma a tabela inteira num ranking dos 10 maiores.

In [ ]:
import pandas as pd

prata = pd.read_csv("etl_northwind/prata/vendas_limpo.csv")

ouro_clientes = (
    prata.groupby(["id_cliente", "cliente", "pais_cliente"])
    .agg(
        receita_total=("receita", "sum"),
        pedidos=("id_pedido", "nunique"),
        itens=("quantidade", "sum"),
    )
    .round(2)
    .reset_index()
    .sort_values("receita_total", ascending=False)
    .head(10)
)

print("Os 10 maiores clientes da Northwind:")
display(ouro_clientes)

print("\nEsses 10 clientes representam",
      round(100 * ouro_clientes["receita_total"].sum() / prata["receita"].sum(), 1),
      "% de toda a receita.")

### ✏️ Atividade Prática 8 — Sua vez de programar

**Contextualização:** a diretoria comercial da Northwind quer decidir em quais países vale a pena abrir um escritório local. O critério não é só quanto o país compra, mas também quantos clientes diferentes existem lá — um país com um único cliente grande é mais arriscado do que um país com vários clientes médios.

**Comando:** abra a camada Prata com `pd.read_csv("etl_northwind/prata/vendas_limpo.csv")`. Agrupe por `pais_cliente` com `groupby()` e crie, com `agg()`, três colunas: `receita_total` (soma de `receita`), `clientes` (usando `"nunique"` sobre `id_cliente`) e `pedidos` (usando `"nunique"` sobre `id_pedido`). Arredonde com `.round(2)`, use `reset_index()`, ordene por `receita_total` do maior para o menor e mostre os 10 primeiros com `display()`.

In [ ]:
# escreva seu código aqui

## Passo 11 — Transformando a camada Ouro em gráfico

A camada Ouro existe justamente para virar gráfico. Com 8 linhas, um gráfico de barras resolve; com 1784, nenhum gráfico ficaria legível — mais uma razão para a agregação vir antes da visualização.

Você já usou o `matplotlib` na Semana 07. A diferença aqui é que o dado não vem de um CSV pronto: vem da camada que o seu próprio pipeline construiu.

### 📖 Antes do código — o gráfico de barras horizontais

`plt.figure(figsize=(largura, altura))` cria a área do gráfico, com o tamanho em polegadas.

`plt.barh(categorias, valores)` desenha barras **horizontais** — a versão deitada do `plt.bar()`. Barras horizontais são a escolha certa quando os rótulos são textos longos, como nomes de categoria: na vertical, esses nomes ficariam sobrepostos ou inclinados.

`plt.gca().invert_yaxis()` inverte o eixo vertical. Sem isso, o maior valor apareceria embaixo, porque o matplotlib desenha o primeiro item na base. A sigla `gca` vem de *get current axes*.

`plt.xlabel("texto")` escreve o rótulo do eixo horizontal e `plt.title("texto")` escreve o título do gráfico. Os dois são da Semana 07, e valem sempre a mesma regra: um gráfico sem rótulo de eixo e sem título obriga quem olha a adivinhar o que está vendo.

`plt.tight_layout()` ajusta as margens para nenhum rótulo ficar cortado, e `plt.show()` exibe o resultado.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

prata = pd.read_csv("etl_northwind/prata/vendas_limpo.csv")

ouro_categoria = (
    prata.groupby("categoria")
    .agg(receita_total=("receita", "sum"))
    .round(2)
    .reset_index()
    .sort_values("receita_total", ascending=False)
)

plt.figure(figsize=(9, 5))
plt.barh(ouro_categoria["categoria"], ouro_categoria["receita_total"], color="#78C043")
plt.gca().invert_yaxis()
plt.xlabel("Receita total (R$)")
plt.title("Receita por categoria de produto — camada Ouro")
plt.tight_layout()
plt.show()

display(ouro_categoria)

### 📖 Antes do código — a evolução mês a mês

Para ver tendência ao longo do tempo, o gráfico certo é o de linha, e o agrupamento certo é por período.

`.dt.to_period("M")` você já usou no Dia 2: ela converte a data no mês a que pertence. `.astype(str)` transforma esse período em texto, que é o formato que o eixo do gráfico entende.

`plt.plot(x, y, marker="o")` desenha a linha ligando os pontos; o `marker="o"` acrescenta uma bolinha em cada mês, deixando visível onde estão os dados reais.

`plt.xticks(rotation=45, ha="right")` gira os rótulos do eixo horizontal em 45 graus e alinha à direita — sem isso, 23 rótulos de mês ficariam ilegíveis, sobrepostos uns aos outros.

`plt.grid(axis="y", alpha=0.3)` acrescenta linhas de grade horizontais discretas (`alpha` é a opacidade, de 0 a 1), que ajudam o olho a comparar alturas. `plt.ylabel("texto")` rotula o eixo vertical, assim como o `xlabel()` fez com o horizontal no gráfico anterior.

No Pandas, `.rename(columns={"antigo": "novo"})` troca o nome de uma coluna. Ele é necessário aqui por um detalhe do `groupby()`: quando você agrupa por uma expressão derivada de uma coluna, o resultado herda o nome da coluna original (`data_pedido`), mesmo o conteúdo sendo outra coisa (o mês). O `rename` corrige esse nome para `mes`, que é o que a coluna realmente contém.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

prata = pd.read_csv("etl_northwind/prata/vendas_limpo.csv")
prata["data_pedido"] = pd.to_datetime(prata["data_pedido"])

ouro_mensal = (
    prata.groupby(prata["data_pedido"].dt.to_period("M").astype(str))
    .agg(receita_total=("receita", "sum"), pedidos=("id_pedido", "nunique"))
    .round(2)
    .reset_index()
    .rename(columns={"data_pedido": "mes"})
)

plt.figure(figsize=(11, 5))
plt.plot(ouro_mensal["mes"], ouro_mensal["receita_total"], marker="o", color="#78C043")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Receita (R$)")
plt.title("Evolução da receita mês a mês — camada Ouro")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Meses na série:", len(ouro_mensal))
display(ouro_mensal.head())

Repare no último ponto da linha: **maio de 1998 despenca**. Isso não é queda de vendas — é o fim do período coberto pelo banco. O último pedido registrado é de 06/05/1998, então esse mês tem só alguns dias de dados.

Esse é um caso clássico de **período incompleto**, e é o tipo de armadilha que faz uma apresentação inteira ir por água abaixo. Quem olha o gráfico sem saber disso conclui que a empresa está quebrando. Num painel de verdade, o mês incompleto é sinalizado ou removido — e a decisão sobre qual dos dois fazer é de quem conhece o negócio.

### ✏️ Atividade Prática 9 — Sua vez de programar

**Contextualização:** a equipe de logística da Northwind quer mostrar para a diretoria que o tempo de entrega varia bastante conforme o destino. Um gráfico comparando os países é mais convincente do que uma tabela cheia de números.

**Comando:** abra a camada Prata com `pd.read_csv()`. Agrupe por `pais_cliente` e calcule a média de `dias_entrega` com `agg(dias_medio=("dias_entrega", "mean"))`, arredondando para 1 casa com `.round(1)`. Use `reset_index()`, ordene do maior para o menor e pegue os 10 primeiros. Desenhe um gráfico de barras horizontais com `plt.barh()`, invertendo o eixo com `plt.gca().invert_yaxis()`, com título e rótulo no eixo. Mostre também a tabela com `display()`.

In [ ]:
# escreva seu código aqui

## Passo 12 — Materializando a camada Ouro em arquivo

Antes de levar o resultado para o banco, a camada Ouro vira arquivo — mesmo procedimento das outras duas camadas.

Aqui são **três** arquivos, um para cada pergunta de negócio: receita por categoria, receita por mês e ranking de clientes. Cada arquivo é uma tabela de destino diferente, e é assim que a camada Ouro costuma ser organizada na prática: uma tabela por indicador, não uma tabela gigante que tenta responder tudo.

In [ ]:
import os
import pandas as pd

prata = pd.read_csv("etl_northwind/prata/vendas_limpo.csv")
prata["data_pedido"] = pd.to_datetime(prata["data_pedido"])

ouro_categoria = (prata.groupby("categoria")
    .agg(receita_total=("receita", "sum"), itens_vendidos=("quantidade", "sum"),
         pedidos=("id_pedido", "nunique"), dias_entrega_medio=("dias_entrega", "mean"))
    .round(2).reset_index().sort_values("receita_total", ascending=False))

ouro_mensal = (prata.groupby(prata["data_pedido"].dt.to_period("M").astype(str))
    .agg(receita_total=("receita", "sum"), pedidos=("id_pedido", "nunique"))
    .round(2).reset_index().rename(columns={"data_pedido": "mes"}))

ouro_clientes = (prata.groupby(["id_cliente", "cliente", "pais_cliente"])
    .agg(receita_total=("receita", "sum"), pedidos=("id_pedido", "nunique"))
    .round(2).reset_index().sort_values("receita_total", ascending=False).head(10))

os.makedirs("etl_northwind/ouro", exist_ok=True)
ouro_categoria.to_csv("etl_northwind/ouro/receita_por_categoria.csv", index=False)
ouro_mensal.to_csv("etl_northwind/ouro/receita_mensal.csv", index=False)
ouro_clientes.to_csv("etl_northwind/ouro/top_clientes.csv", index=False)

print("Três arquivos da camada Ouro gravados. Confirmando pela releitura:\n")
for nome in ["receita_por_categoria", "receita_mensal", "top_clientes"]:
    tabela = pd.read_csv(f"etl_northwind/ouro/{nome}.csv")
    print(f"{nome}.csv -> {len(tabela)} linhas, {len(tabela.columns)} colunas")

print("\nReceita por categoria (arquivo reaberto do disco):")
display(pd.read_csv("etl_northwind/ouro/receita_por_categoria.csv"))

### 👀 Veja no VS Code: as três camadas completas

Abra o Explorer (**Ctrl + Shift + E**) e expanda `etl_northwind`. A estrutura completa do pipeline está lá:

```
etl_northwind/
├── bronze/
│   ├── vendas_bruto.csv          2155 linhas — o dado como veio
│   └── metadados_extracao.csv
├── prata/
│   └── vendas_limpo.csv          1784 linhas — limpo, com colunas novas
└── ouro/
    ├── receita_por_categoria.csv    8 linhas
    ├── receita_mensal.csv          23 linhas
    └── top_clientes.csv            10 linhas
```

Abra os três arquivos da pasta `ouro/`. Repare no tamanho deles comparado ao da Bronze: o `receita_por_categoria.csv` tem 9 linhas no total (8 categorias mais o cabeçalho) e cabe inteiro na tela. É esse arquivo que vira o gráfico da apresentação — não os 2155 registros do começo.

Esse encolhimento é o resumo visual do pipeline inteiro: **2155 → 1784 → 8**. Cada seta dessas é uma decisão que você tomou e documentou.

In [ ]:
import os

print("Pipeline completo em disco:\n")

for pasta_atual, subpastas, arquivos in os.walk("etl_northwind"):
    nivel = pasta_atual.replace("etl_northwind", "").count(os.sep)
    recuo = "    " * nivel
    print(f"{recuo}{os.path.basename(pasta_atual) or 'etl_northwind'}/")
    for arquivo in sorted(arquivos):
        caminho = os.path.join(pasta_atual, arquivo)
        tamanho_kb = os.path.getsize(caminho) / 1024
        print(f"{recuo}    {arquivo}  ({tamanho_kb:.1f} KB)")

## Passo 13 — Load: levando a camada Ouro para o PostgreSQL

Chegou o **L** do ETL. Os arquivos resolvem para quem usa Python, mas a maioria das ferramentas de painel (Power BI, Metabase, Looker) se conecta a um **banco**, não a uma pasta de CSVs. Além disso, um banco controla acesso, tipo de dado e concorrência — coisas que um arquivo solto não faz.

**Onde carregar?** Não nas tabelas `clientes`, `pedidos` e companhia: aquelas são do sistema de vendas, e um processo analítico nunca escreve no banco de operação. A solução padrão é criar um **schema** separado.

Um schema no PostgreSQL é como uma pasta dentro do banco: agrupa tabelas sob um nome e mantém tudo separado. O banco continua sendo o `northwind`; dentro dele, `public` guarda o sistema de vendas e `analytics` vai guardar o resultado do seu ETL.

### 📖 Antes do código — `CREATE SCHEMA` e `CREATE TABLE`

`CREATE SCHEMA IF NOT EXISTS analytics` cria o schema. O `IF NOT EXISTS` evita erro se ele já existir — o que importa aqui porque um ETL roda várias vezes.

`CREATE TABLE IF NOT EXISTS analytics.ouro_receita_categoria (...)` cria a tabela dentro do schema. Cada coluna precisa de um tipo, e a escolha deles é uma decisão de qualidade:

| Tipo | Quando usar | Por que aqui |
|---|---|---|
| `VARCHAR(50)` | texto com tamanho máximo previsível | nome de categoria |
| `NUMERIC(12,2)` | valor em dinheiro | guarda exatamente 2 casas, **sem** o resíduo de ponto flutuante do Dia 2 |
| `INTEGER` | contagem | itens vendidos, pedidos |
| `NUMERIC(5,1)` | média com 1 casa | dias de entrega |

O `NUMERIC(12,2)` merece destaque: diferente do tipo de ponto flutuante, ele guarda o valor decimal de forma exata. É por isso que todo sistema financeiro usa `NUMERIC` (ou `DECIMAL`, que é sinônimo) para dinheiro.

`conexao.commit()` confirma as alterações. Sem ele, tudo que o cursor executou é desfeito quando a conexão fecha — comportamento que você já viu na Semana 11.

In [ ]:
import psycopg2

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor = conexao.cursor()

cursor.execute("CREATE SCHEMA IF NOT EXISTS analytics")

cursor.execute('''
    CREATE TABLE IF NOT EXISTS analytics.ouro_receita_categoria (
        categoria           VARCHAR(50) PRIMARY KEY,
        receita_total       NUMERIC(12,2),
        itens_vendidos      INTEGER,
        pedidos             INTEGER,
        dias_entrega_medio  NUMERIC(5,1)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS analytics.ouro_receita_mensal (
        mes             VARCHAR(7) PRIMARY KEY,
        receita_total   NUMERIC(12,2),
        pedidos         INTEGER
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS analytics.ouro_top_clientes (
        id_cliente      VARCHAR(5) PRIMARY KEY,
        cliente         VARCHAR(100),
        pais            VARCHAR(50),
        receita_total   NUMERIC(12,2),
        pedidos         INTEGER
    )
''')

conexao.commit()

cursor.execute('''
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'analytics' ORDER BY table_name
''')
print("Tabelas criadas no schema analytics:")
for linha in cursor.fetchall():
    print(" -", linha[0])

conexao.close()

### 👀 Veja as tabelas sem sair do VS Code

As três tabelas existem no banco agora — e você não precisa abrir o pgAdmin para conferir. Dá para enxergar a estrutura do banco pelo próprio notebook, e é isso que a célula abaixo faz.

### 📖 Antes do código — lendo o catálogo do schema `analytics`

A consulta junta duas tabelas do catálogo que você já conhece do Dia 1: `information_schema.tables` (quais tabelas existem) e `information_schema.columns` (quais colunas cada uma tem). Juntando as duas, você monta exatamente a árvore que o pgAdmin mostra na lateral.

Três colunas do catálogo aparecem aqui pela primeira vez:

- `ordinal_position` — a posição da coluna dentro da tabela (1ª, 2ª, 3ª...), que é a ordem em que ela foi declarada no `CREATE TABLE`.
- `data_type` — o tipo da coluna, do jeito que o PostgreSQL guarda internamente (`character varying` é o nome completo de `VARCHAR`, e `timestamp without time zone` é o do `TIMESTAMP`).
- `character_maximum_length` e `numeric_precision`/`numeric_scale` — o tamanho declarado. O primeiro vale para texto (`VARCHAR(50)` → 50) e os outros dois para número (`NUMERIC(12,2)` → precisão 12, escala 2).

`COALESCE(a, b, c)` devolve o primeiro valor que não for nulo, entre os que você listar. Como uma coluna de texto não tem precisão numérica e uma coluna numérica não tem comprimento de texto, o `COALESCE` escolhe automaticamente qual dos dois mostrar.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

estrutura = pd.read_sql('''
    SELECT t.table_name AS tabela,
           c.ordinal_position AS posicao,
           c.column_name AS coluna,
           c.data_type AS tipo,
           COALESCE(
               c.character_maximum_length::text,
               c.numeric_precision::text || ',' || c.numeric_scale::text,
               '-'
           ) AS tamanho
    FROM information_schema.tables t
    JOIN information_schema.columns c
      ON c.table_schema = t.table_schema
     AND c.table_name = t.table_name
    WHERE t.table_schema = 'analytics'
    ORDER BY t.table_name, c.ordinal_position
''', conexao)

conexao.close()

print("Estrutura do schema analytics —", estrutura["tabela"].nunique(), "tabelas,",
      len(estrutura), "colunas no total:")
display(estrutura)

Essa é a mesma informação que o pgAdmin mostra quando você expande **Tables → Columns**, só que dentro do notebook, ao lado do código que criou essas tabelas.

Mais para o fim deste encontro, no Passo 21, você vai instalar uma extensão e ver essa mesma estrutura numa árvore na barra lateral do VS Code, sem abrir o pgAdmin. Até lá, esta consulta resolve — e ela continua útil depois, porque roda em qualquer máquina, sem instalar nada.

### 📖 Antes do código — `execute_values()` e a carga em lote

Inserir 8 linhas com 8 comandos `INSERT` separados funciona, mas é lento: cada comando é uma viagem de ida e volta até o banco. Com 100 mil linhas, isso levaria minutos.

`execute_values(cursor, comando, lista_de_tuplas)` resolve isso: ela monta **um único** `INSERT` com todos os valores de uma vez. A função vem do módulo `psycopg2.extras`, por isso o import separado.

O comando leva um `%s` sozinho no lugar da lista de valores — é ali que a função encaixa todas as linhas. Esse `%s` é o mesmo marcador de parâmetro que você usou na Semana 11 para evitar SQL Injection: o valor nunca é colado no texto do comando, ele viaja separado.

`df.itertuples(index=False, name=None)` percorre o DataFrame devolvendo cada linha como uma tupla simples, sem o índice — que é exatamente o formato que a função espera.

O `try`/`except` em volta da inserção tem um motivo específico: se você já tiver rodado esta célula antes (numa aula anterior, ou repetindo o notebook), a tabela não está mais vazia, e o banco vai recusar a segunda inserção. Em vez de quebrar, a célula avisa e manda você seguir para o Passo 14 — que é justamente sobre esse problema.

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
import pandas as pd

ouro_categoria = pd.read_csv("etl_northwind/ouro/receita_por_categoria.csv")

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor = conexao.cursor()

linhas = list(ouro_categoria.itertuples(index=False, name=None))
print("Primeira linha que será enviada:", linhas[0])

try:
    execute_values(cursor, '''
        INSERT INTO analytics.ouro_receita_categoria
            (categoria, receita_total, itens_vendidos, pedidos, dias_entrega_medio)
        VALUES %s
    ''', linhas)
    conexao.commit()
    print("Carga concluída:", len(linhas), "linhas inseridas.")
except psycopg2.errors.UniqueViolation:
    conexao.rollback()
    print("A tabela já tinha esses dados — ou seja, esta célula já rodou antes.")
    print("Isso é o assunto do Passo 14; siga em frente que o problema é explicado lá.")

conferencia = pd.read_sql("SELECT * FROM analytics.ouro_receita_categoria ORDER BY receita_total DESC", conexao)
print("\nLinhas agora dentro do PostgreSQL:", len(conferencia))
display(conferencia)

conexao.close()

## Passo 14 — O que acontece se o ETL rodar duas vezes?

Essa é a pergunta que separa um script de um pipeline. Um ETL de verdade **roda de novo**: toda madrugada, ou porque a execução de ontem falhou no meio, ou porque alguém corrigiu uma regra.

A célula abaixo simula exatamente isso: executa a mesma carga do Passo 13 outra vez, sem mudar nada.

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
import pandas as pd

ouro_categoria = pd.read_csv("etl_northwind/ouro/receita_por_categoria.csv")

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor = conexao.cursor()

cursor.execute("SELECT COUNT(*), SUM(receita_total) FROM analytics.ouro_receita_categoria")
antes = cursor.fetchone()
print("ANTES de rodar de novo -> linhas:", antes[0], "| receita:", antes[1])

try:
    execute_values(cursor, '''
        INSERT INTO analytics.ouro_receita_categoria
            (categoria, receita_total, itens_vendidos, pedidos, dias_entrega_medio)
        VALUES %s
    ''', list(ouro_categoria.itertuples(index=False, name=None)))
    conexao.commit()
    print("\nA carga rodou sem erro nenhum.")
except psycopg2.errors.UniqueViolation as erro:
    conexao.rollback()
    print("\nO banco recusou a segunda carga, e é este o motivo:")
    print(str(erro).strip())

cursor.execute("SELECT COUNT(*), SUM(receita_total) FROM analytics.ouro_receita_categoria")
depois = cursor.fetchone()
print("\nDEPOIS -> linhas:", depois[0], "| receita:", depois[1])

conexao.close()

O banco recusou a segunda carga com o erro `duplicate key value violates unique constraint` — em português, "valor de chave duplicado viola a restrição de unicidade".

Quem impediu o estrago foi a `PRIMARY KEY` que você declarou na coluna `categoria` lá no `CREATE TABLE`. Sem ela, a segunda carga teria entrado normalmente, e a tabela ficaria com 16 linhas e o dobro da receita — um relatório com o valor errado, sem nenhuma mensagem de erro para avisar.

Pense no tamanho desse problema: a diretoria receberia um painel dizendo que a Northwind faturou R$ 2 milhões em vez de R$ 1 milhão, e ninguém teria como perceber olhando o painel.

**A solução é tornar a carga idempotente.** Idempotente é um processo que, rodado várias vezes, produz sempre o mesmo resultado — como apertar o botão do elevador dez vezes: ele sobe uma vez só.

### 📖 Antes do código — `TRUNCATE` e a recarga completa

`TRUNCATE TABLE nome` apaga **todas** as linhas da tabela, mantendo a estrutura (colunas, tipos, chaves). É mais rápido que `DELETE FROM`, porque não remove linha por linha.

A estratégia "apagar tudo e inserir tudo de novo" se chama **recarga completa** (*full refresh*), e é a escolha certa para tabelas agregadas pequenas como as da camada Ouro: as três juntas têm 41 linhas. Para uma tabela de milhões de linhas a escolha seria outra (carregar só o que mudou), mas aqui a simplicidade vence.

O ponto crítico é que o `TRUNCATE` e o `INSERT` precisam estar na **mesma transação**. Como o `commit()` só aparece depois dos dois, o banco trata o par como uma operação única: ou os dois acontecem, ou nenhum acontece. Sem isso, uma falha entre o `TRUNCATE` e o `INSERT` deixaria a tabela vazia.

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
import pandas as pd

ouro_categoria = pd.read_csv("etl_northwind/ouro/receita_por_categoria.csv")

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor = conexao.cursor()

for tentativa in [1, 2, 3]:
    cursor.execute("TRUNCATE TABLE analytics.ouro_receita_categoria")
    execute_values(cursor, '''
        INSERT INTO analytics.ouro_receita_categoria
            (categoria, receita_total, itens_vendidos, pedidos, dias_entrega_medio)
        VALUES %s
    ''', list(ouro_categoria.itertuples(index=False, name=None)))
    conexao.commit()

    cursor.execute("SELECT COUNT(*), SUM(receita_total) FROM analytics.ouro_receita_categoria")
    resultado = cursor.fetchone()
    print(f"Execução {tentativa} -> linhas: {resultado[0]} | receita: {resultado[1]}")

print("\nTrês execuções, o mesmo resultado. A carga é idempotente.")
conexao.close()

## Passo 15 — Quando a carga falha no meio do caminho

Rodar três vezes e dar o mesmo resultado resolve a repetição. Falta o outro cenário: a execução que **quebra no meio** — a conexão cai, o servidor reinicia, um valor inesperado aparece na linha 5000.

A pergunta é: o que sobra no banco quando isso acontece?

### 📖 Antes do código — transação, `commit()` e `rollback()`

Uma **transação** é um bloco de comandos que o banco trata como uma coisa só. Ela tem dois finais possíveis:

- `conexao.commit()` — confirma tudo que foi feito desde o início da transação.
- `conexao.rollback()` — desfaz tudo, deixando o banco exatamente como estava antes.

No psycopg2 a transação começa sozinha no primeiro comando e só termina quando você chama um dos dois. É por isso que o `commit()` sempre apareceu no fim dos exemplos.

O código abaixo provoca uma falha de propósito, no meio da carga: depois do `TRUNCATE` e da primeira inserção, um `raise` interrompe o processo. `raise` é o comando que lança um erro manualmente — serve para simular exatamente a queda que aconteceria num dia ruim.

O `except` captura essa falha e chama `rollback()`. O resultado é o que interessa: a tabela volta a ter as 8 linhas da carga anterior, e não fica nem vazia nem pela metade.

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
import pandas as pd

ouro_categoria = pd.read_csv("etl_northwind/ouro/receita_por_categoria.csv")

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor = conexao.cursor()

cursor.execute("SELECT COUNT(*) FROM analytics.ouro_receita_categoria")
print("Linhas na tabela ANTES da carga com falha:", cursor.fetchone()[0])

try:
    cursor.execute("TRUNCATE TABLE analytics.ouro_receita_categoria")

    metade = ouro_categoria.head(4)
    execute_values(cursor, '''
        INSERT INTO analytics.ouro_receita_categoria
            (categoria, receita_total, itens_vendidos, pedidos, dias_entrega_medio)
        VALUES %s
    ''', list(metade.itertuples(index=False, name=None)))
    print("4 linhas inseridas... e agora a conexão vai cair.")

    raise ConnectionError("simulação: a rede caiu no meio da carga")

except ConnectionError as erro:
    conexao.rollback()
    print("Falha capturada:", erro)
    print("Rollback executado — nada do que esta carga fez foi mantido.")

cursor.execute("SELECT COUNT(*) FROM analytics.ouro_receita_categoria")
print("\nLinhas na tabela DEPOIS da falha:", cursor.fetchone()[0])
print("A tabela continua com a carga íntegra da execução anterior.")

conexao.close()

Esse é o comportamento que você quer: **ou a carga inteira acontece, ou nada acontece**. Sem a transação, a tabela teria ficado com 4 categorias — e um relatório com 4 de 8 categorias parece perfeitamente normal para quem olha de fora. Ninguém receberia mensagem de erro; só um número menor do que deveria.

Vale guardar a frase que resume o Passo 14 e o 15 juntos: **o pior resultado de um ETL não é o erro que aparece — é o erro que não aparece.**

## Passo 16 — Validação: o código rodou, mas o resultado está certo?

A carga terminou sem erro. Isso significa que está correta? Não necessariamente. Executar sem erro e produzir o resultado certo são coisas diferentes: a consulta pode ter trazido menos linhas do que devia, um `JOIN` pode ter multiplicado registros, um filtro pode ter cortado demais.

Validar significa **comparar origem e destino** por conta própria, com números que você calcula dos dois lados.

### 📖 Antes do código — as três checagens

O código faz três conferências independentes e mostra o resultado de cada uma:

| Checagem | O que compara | Por que importa |
|---|---|---|
| Contagem | linhas no CSV × linhas na tabela | pega linha perdida no caminho |
| Soma | receita no CSV × receita na tabela | pega valor truncado ou tipo errado |
| Conteúdo | categorias no CSV × categorias na tabela | pega registro trocado, não só faltando |

`float(...)` aparece na comparação de soma porque o PostgreSQL devolve `NUMERIC` como um objeto `Decimal`, e comparar `Decimal` com o `float` do Pandas diretamente pode falhar por diferença de tipo. Converter os dois para `float` antes de comparar resolve.

`abs(a - b) < 0.01` compara com uma tolerância de um centavo, em vez de exigir igualdade exata. Essa é a forma correta de comparar valores decimais depois de conversões entre sistemas diferentes.

In [ ]:
import psycopg2
import pandas as pd

ouro_categoria = pd.read_csv("etl_northwind/ouro/receita_por_categoria.csv")

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor = conexao.cursor()

cursor.execute("SELECT COUNT(*), SUM(receita_total) FROM analytics.ouro_receita_categoria")
linhas_destino, receita_destino = cursor.fetchone()

linhas_origem = len(ouro_categoria)
receita_origem = ouro_categoria["receita_total"].sum()

cursor.execute("SELECT categoria FROM analytics.ouro_receita_categoria ORDER BY categoria")
categorias_destino = sorted(linha[0] for linha in cursor.fetchall())
categorias_origem = sorted(ouro_categoria["categoria"].tolist())

checagens = [
    {"checagem": "Contagem de linhas", "origem": linhas_origem, "destino": linhas_destino,
     "resultado": "OK" if linhas_origem == linhas_destino else "FALHOU"},
    {"checagem": "Soma da receita", "origem": round(receita_origem, 2), "destino": float(receita_destino),
     "resultado": "OK" if abs(float(receita_destino) - receita_origem) < 0.01 else "FALHOU"},
    {"checagem": "Categorias iguais", "origem": len(categorias_origem), "destino": len(categorias_destino),
     "resultado": "OK" if categorias_origem == categorias_destino else "FALHOU"},
]

display(pd.DataFrame(checagens))

if all(item["resultado"] == "OK" for item in checagens):
    print("Todas as checagens passaram: a carga está validada.")
else:
    print("Alguma checagem falhou — a carga NÃO pode ser considerada concluída.")

conexao.close()

## Passo 17 — Rastreabilidade: deixando registro do que aconteceu

Última peça. Depois que o ETL roda, alguém vai perguntar: quando foi a última execução? Quantas linhas entraram? Deu certo?

Responder isso de memória não escala. A solução é o pipeline **registrar a própria execução** numa tabela de log.

### 📖 Antes do código — a tabela de log

A tabela `analytics.etl_execucao` guarda uma linha por execução, com o que aconteceu.

`SERIAL PRIMARY KEY` é um tipo especial do PostgreSQL: a cada inserção, ele gera automaticamente o próximo número inteiro. Você não precisa informar o `id` — o banco cuida disso. É o equivalente automático ao `MAX(id) + 1` que você fez na mão no bônus da Semana 11.

`TIMESTAMP` guarda data **e** hora, diferente de `DATE`, que guarda só a data.

`cursor.execute(comando, (valor1, valor2, ...))` com a tupla separada é o formato seguro de passar parâmetros, o mesmo da Semana 11: os `%s` do comando são preenchidos pelo driver, não pela formatação de texto do Python.

In [ ]:
import psycopg2
import pandas as pd
from datetime import datetime

ouro_categoria = pd.read_csv("etl_northwind/ouro/receita_por_categoria.csv")

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor = conexao.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS analytics.etl_execucao (
        id              SERIAL PRIMARY KEY,
        executado_em    TIMESTAMP,
        camada          VARCHAR(30),
        linhas_origem   INTEGER,
        linhas_destino  INTEGER,
        status          VARCHAR(20)
    )
''')

cursor.execute("SELECT COUNT(*) FROM analytics.ouro_receita_categoria")
linhas_destino = cursor.fetchone()[0]
linhas_origem = len(ouro_categoria)
status = "SUCESSO" if linhas_origem == linhas_destino else "DIVERGENTE"

cursor.execute('''
    INSERT INTO analytics.etl_execucao
        (executado_em, camada, linhas_origem, linhas_destino, status)
    VALUES (%s, %s, %s, %s, %s)
''', (datetime.now(), "ouro_receita_categoria", linhas_origem, linhas_destino, status))

conexao.commit()

historico = pd.read_sql("SELECT * FROM analytics.etl_execucao ORDER BY id DESC", conexao)
print("Histórico de execuções do ETL:")
display(historico)

conexao.close()

Rode essa célula mais de uma vez e veja o histórico crescer. Essa tabela é o que permite, daqui a três meses, responder "o ETL rodou ontem?" sem depender da memória de ninguém — e é ela que um sistema de monitoramento consultaria para disparar um alerta quando o status voltasse `DIVERGENTE`.

**Uma observação sobre esta tabela:** repare que ela é a única do schema `analytics` que **não** é truncada a cada execução. Faz sentido: o histórico só tem valor se acumular. Cada tabela do pipeline tem a sua própria política, e confundir as duas — truncar o log, ou acumular a tabela de indicadores — quebra o processo de um jeito ou de outro.

## Passo 18 — O pipeline inteiro, numa função só

Tudo que você construiu em três dias está espalhado em dezenas de células. Num projeto real, isso vira **um** arquivo que roda sozinho, sem ninguém clicando em célula nenhuma.

### 📖 Antes do código — organizando em funções

Cada etapa do ETL vira uma função, com uma responsabilidade só:

| Função | Responsabilidade |
|---|---|
| `extrair()` | conecta na origem e devolve a camada Bronze |
| `transformar(bronze)` | aplica as 5 regras e devolve a camada Prata |
| `agregar(prata)` | devolve as três tabelas da camada Ouro |
| `carregar(tabela, nome, colunas)` | grava uma tabela Ouro no PostgreSQL, de forma idempotente |
| `validar(tabela, nome)` | compara origem e destino e devolve se passou |
| `registrar(...)` | grava o resultado da execução na tabela de log |
| `executar_pipeline()` | chama todas na ordem certa |

Essa separação você já aprendeu na Semana 04: uma função recebe dados, faz **uma** coisa e devolve o resultado. O ganho aqui é concreto — se a regra de limpeza mudar, você mexe só em `transformar()`, e o resto continua igual.

Duas novidades de sintaxe. `psycopg2.connect(**CONFIGURACAO)` usa `**` para desempacotar um dicionário em argumentos nomeados: em vez de repetir host, dbname, user e password em cada função, a configuração fica num lugar só. E o bloco `finally` de um `try` executa sempre, dando certo ou errado — é o lugar correto para fechar a conexão, garantindo que ela não fique aberta nem quando a carga falha.

In [ ]:
import os
from datetime import datetime

import psycopg2
from psycopg2.extras import execute_values
import pandas as pd

CONFIGURACAO = {
    "host": "localhost",
    "dbname": "northwind",
    "user": "postgres",
    "password": "SUA_SENHA_AQUI",
}


def extrair():
    conexao = psycopg2.connect(**CONFIGURACAO)
    bronze = pd.read_sql('''
        SELECT p.id_pedido, p.data_pedido, p.required_date AS data_prometida, p.data_envio,
               p.freight AS frete, p.ship_country AS pais_entrega, p.ship_region AS regiao_entrega,
               cl.id_cliente, cl.nome_empresa AS cliente, cl.cidade AS cidade_cliente,
               cl.pais AS pais_cliente, pr.nome_produto AS produto, pr.descontinuado,
               cat.nome_categoria AS categoria, ip.preco_unitario, ip.quantidade, ip.desconto
        FROM itens_pedido ip
        JOIN pedidos p ON p.id_pedido = ip.id_pedido
        JOIN clientes cl ON cl.id_cliente = p.id_cliente
        JOIN produtos pr ON pr.id_produto = ip.id_produto
        JOIN categorias cat ON cat.id_categoria = pr.id_categoria
    ''', conexao)
    conexao.close()

    os.makedirs("etl_northwind/bronze", exist_ok=True)
    bronze.to_csv("etl_northwind/bronze/vendas_bruto.csv", index=False)
    return bronze


def transformar(bronze):
    for coluna in ["data_pedido", "data_prometida", "data_envio"]:
        bronze[coluna] = pd.to_datetime(bronze[coluna])

    prata = bronze.dropna(subset=["data_envio"]).copy()
    prata["regiao_entrega"] = prata["regiao_entrega"].fillna("NAO INFORMADO")
    prata = prata[prata["descontinuado"] == 0].copy()
    prata["receita"] = (prata["preco_unitario"] * prata["quantidade"] * (1 - prata["desconto"])).round(2)
    prata["dias_entrega"] = (prata["data_envio"] - prata["data_pedido"]).dt.days
    prata["entrega_atrasada"] = prata["data_envio"] > prata["data_prometida"]

    os.makedirs("etl_northwind/prata", exist_ok=True)
    prata.to_csv("etl_northwind/prata/vendas_limpo.csv", index=False)
    return prata


def agregar(prata):
    categoria = (prata.groupby("categoria")
        .agg(receita_total=("receita", "sum"), itens_vendidos=("quantidade", "sum"),
             pedidos=("id_pedido", "nunique"), dias_entrega_medio=("dias_entrega", "mean"))
        .round(2).reset_index().sort_values("receita_total", ascending=False))

    mensal = (prata.groupby(prata["data_pedido"].dt.to_period("M").astype(str))
        .agg(receita_total=("receita", "sum"), pedidos=("id_pedido", "nunique"))
        .round(2).reset_index().rename(columns={"data_pedido": "mes"}))

    clientes = (prata.groupby(["id_cliente", "cliente", "pais_cliente"])
        .agg(receita_total=("receita", "sum"), pedidos=("id_pedido", "nunique"))
        .round(2).reset_index().sort_values("receita_total", ascending=False).head(10))

    os.makedirs("etl_northwind/ouro", exist_ok=True)
    categoria.to_csv("etl_northwind/ouro/receita_por_categoria.csv", index=False)
    mensal.to_csv("etl_northwind/ouro/receita_mensal.csv", index=False)
    clientes.to_csv("etl_northwind/ouro/top_clientes.csv", index=False)

    return {
        "ouro_receita_categoria": categoria,
        "ouro_receita_mensal": mensal,
        "ouro_top_clientes": clientes,
    }


def carregar(tabela, nome_no_banco, colunas):
    conexao = psycopg2.connect(**CONFIGURACAO)
    cursor = conexao.cursor()
    try:
        cursor.execute(f"TRUNCATE TABLE analytics.{nome_no_banco}")
        execute_values(cursor,
            f"INSERT INTO analytics.{nome_no_banco} ({colunas}) VALUES %s",
            list(tabela.itertuples(index=False, name=None)))
        conexao.commit()
        return True
    except Exception as erro:
        conexao.rollback()
        print("      Falha ao carregar", nome_no_banco, "->", erro)
        return False
    finally:
        conexao.close()


def validar(tabela, nome_no_banco):
    conexao = psycopg2.connect(**CONFIGURACAO)
    cursor = conexao.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM analytics.{nome_no_banco}")
    linhas_destino = cursor.fetchone()[0]
    conexao.close()
    return len(tabela) == linhas_destino, linhas_destino


def registrar(camada, linhas_origem, linhas_destino, status):
    conexao = psycopg2.connect(**CONFIGURACAO)
    cursor = conexao.cursor()
    cursor.execute('''
        INSERT INTO analytics.etl_execucao
            (executado_em, camada, linhas_origem, linhas_destino, status)
        VALUES (%s, %s, %s, %s, %s)
    ''', (datetime.now(), camada, linhas_origem, linhas_destino, status))
    conexao.commit()
    conexao.close()


def executar_pipeline():
    print("[1/4] Extraindo...")
    bronze = extrair()
    print("      Bronze:", len(bronze), "linhas")

    print("[2/4] Transformando...")
    prata = transformar(bronze)
    print("      Prata:", len(prata), "linhas")

    print("[3/4] Agregando...")
    tabelas_ouro = agregar(prata)
    for nome, tabela in tabelas_ouro.items():
        print(f"      {nome}: {len(tabela)} linhas")

    print("[4/4] Carregando e validando...")
    colunas_por_tabela = {
        "ouro_receita_categoria": "categoria, receita_total, itens_vendidos, pedidos, dias_entrega_medio",
        "ouro_receita_mensal": "mes, receita_total, pedidos",
        "ouro_top_clientes": "id_cliente, cliente, pais, receita_total, pedidos",
    }

    for nome, tabela in tabelas_ouro.items():
        carregou = carregar(tabela, nome, colunas_por_tabela[nome])
        validou, linhas_destino = validar(tabela, nome)
        status = "SUCESSO" if carregou and validou else "FALHOU"
        registrar(nome, len(tabela), linhas_destino, status)
        print(f"      {nome}: {status} ({linhas_destino} linhas no banco)")

    print("\nPipeline concluído.")


executar_pipeline()

Uma célula, o pipeline inteiro. E repare no que a saída mostra: 2155 → 1784 → as três tabelas Ouro, cada uma carregada, validada e registrada no log.

Esse código é, na prática, o produto final da sua semana. Copiado para um arquivo `.py`, ele roda pelo terminal com `python pipeline_northwind.py`, sem VS Code e sem notebook — que é exatamente como um ETL roda em produção, agendado para as 3 da manhã.

O resultado final agora vive em dois lugares: os arquivos `.csv` em disco e as tabelas dentro do PostgreSQL. O mesmo dado, dois formatos, cada um servindo a um público — o CSV para quem vai analisar em Python, a tabela para quem vai conectar um painel.

No próximo passo você completa o medalhão dentro do banco, e no Passo 20 monta um jeito de enxergar tudo **sem sair do VS Code**.

## Passo 19 — O medalhão completo dentro do banco

Repare numa coisa: até agora só a camada **Ouro** virou tabela no PostgreSQL. A Bronze e a Prata existem apenas como arquivo `.csv`.

Isso funciona, mas tem um limite prático. Quem quiser conferir de onde saiu um número da camada Ouro precisa abrir um CSV de 2155 linhas no VS Code e procurar com os olhos — não dá para escrever `SELECT ... WHERE` num arquivo de texto.

Levar as três camadas para o banco resolve isso e traz duas vantagens concretas:

- **Rastreabilidade com SQL:** dá para partir de um número da Ouro e descer até as linhas da Prata e da Bronze que o originaram, com uma consulta.
- **Reprocessamento sem tocar na origem:** se uma regra de limpeza mudar, você reconstrói a Prata a partir da tabela Bronze do banco, sem precisar consultar de novo o sistema de vendas.

É por isso que, na prática, a arquitetura medalhão quase sempre vive **inteira** no banco — é o que você vai fazer agora.

### 📖 Antes do código — as tabelas das duas camadas

Nenhuma função nova aqui: é o mesmo `CREATE TABLE IF NOT EXISTS` do Passo 13, agora com mais colunas.

Duas escolhas de tipo merecem atenção:

- `DATE` nas colunas de data — e não `VARCHAR`. Guardando como data de verdade, o banco permite `WHERE data_pedido > '1998-01-01'` e ordenação cronológica correta. Guardado como texto, `'1998-01-01'` viria antes de `'1997-12-31'` na ordenação alfabética.
- `BOOLEAN` em `entrega_atrasada` — o tipo próprio para verdadeiro/falso, que aceita `TRUE`/`FALSE` e permite `WHERE entrega_atrasada` direto, sem comparação.

Repare também que a tabela `bronze_vendas` **não** tem `PRIMARY KEY`. Isso é deliberado: a camada Bronze é uma cópia fiel da origem, e o grão dela (item de pedido) tem `id_pedido` repetido em várias linhas. Declarar chave primária ali quebraria a carga — e a Bronze não precisa dessa proteção, porque ela é sempre recarregada por inteiro.

In [ ]:
import psycopg2

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor = conexao.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS analytics.bronze_vendas (
        id_pedido        INTEGER,
        data_pedido      DATE,
        data_prometida   DATE,
        data_envio       DATE,
        frete            NUMERIC(10,2),
        pais_entrega     VARCHAR(50),
        regiao_entrega   VARCHAR(50),
        id_cliente       VARCHAR(5),
        cliente          VARCHAR(100),
        cidade_cliente   VARCHAR(50),
        pais_cliente     VARCHAR(50),
        produto          VARCHAR(100),
        descontinuado    INTEGER,
        categoria        VARCHAR(50),
        preco_unitario   NUMERIC(10,2),
        quantidade       INTEGER,
        desconto         NUMERIC(5,2)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS analytics.prata_vendas (
        id_pedido        INTEGER,
        data_pedido      DATE,
        data_prometida   DATE,
        data_envio       DATE,
        frete            NUMERIC(10,2),
        pais_entrega     VARCHAR(50),
        regiao_entrega   VARCHAR(50),
        id_cliente       VARCHAR(5),
        cliente          VARCHAR(100),
        cidade_cliente   VARCHAR(50),
        pais_cliente     VARCHAR(50),
        produto          VARCHAR(100),
        descontinuado    INTEGER,
        categoria        VARCHAR(50),
        preco_unitario   NUMERIC(10,2),
        quantidade       INTEGER,
        desconto         NUMERIC(5,2),
        receita          NUMERIC(12,2),
        dias_entrega     INTEGER,
        entrega_atrasada BOOLEAN
    )
''')

conexao.commit()
print("Tabelas bronze_vendas e prata_vendas criadas.")
conexao.close()

### ⚠️ Um erro real: o vazio do Pandas não é o vazio do SQL

Agora a carga da camada Bronze — e ela vai falhar na primeira tentativa, de propósito.

A camada Bronze tem 73 linhas com `data_envio` vazia. No Pandas, esse vazio é representado por `NaN` (*Not a Number*). No banco, o vazio se chama `NULL`. São duas coisas diferentes, e o `psycopg2` não traduz uma na outra sozinho.

A célula abaixo tenta a carga sem tratar isso.

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
import pandas as pd

bronze = pd.read_csv("etl_northwind/bronze/vendas_bruto.csv")
com_data_vazia = bronze[bronze["data_envio"].isna()]
print("Linhas com data_envio vazia:", len(com_data_vazia))

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor = conexao.cursor()

try:
    execute_values(cursor,
        "INSERT INTO analytics.bronze_vendas VALUES %s",
        list(com_data_vazia.head(5).itertuples(index=False, name=None)))
    conexao.commit()
    print("Inseriu (não deveria!)")
except psycopg2.errors.DatatypeMismatch as erro:
    conexao.rollback()
    print("\nDeu erro, e é este aqui:")
    print(str(erro).strip())

conexao.close()

A mensagem é direta: `coluna "data_envio" é do tipo date mas expressão é do tipo double precision`.

O que aconteceu: o `NaN` do Pandas é tecnicamente um **número decimal** (por isso *double precision*, o nome do tipo de número com casas decimais no PostgreSQL). O banco recebeu um número onde esperava uma data, e recusou.

Esse é um dos erros mais comuns de quem carrega DataFrame em banco, e a boa notícia é que ele **aparece** — o banco recusa em vez de gravar lixo. Nem todo erro é assim gentil.

### 📖 Antes do código — traduzindo `NaN` para `NULL`

A correção é converter os vazios do Pandas em `None`, que é o vazio do Python — e esse o `psycopg2` traduz para `NULL` do SQL automaticamente.

`pd.notna(tabela)` devolve um DataFrame de `True`/`False`, com `True` onde há valor preenchido (é o oposto do `isna()` que você usou no Dia 1).

`tabela.astype(object)` converte as colunas para o tipo genérico do Python. Esse passo é necessário porque uma coluna numérica não aceita guardar `None` — ela converteria de volta para `NaN`. Com o tipo `object`, a coluna passa a aceitar qualquer coisa, inclusive `None`.

`.where(condicao, None)` mantém o valor onde a condição é `True` e troca por `None` onde é `False`. Juntando tudo: mantém o que está preenchido, troca o vazio por `None`.

O resultado é uma linha só, e vale guardar como receita para qualquer carga de DataFrame em banco:

```python
pronto = tabela.astype(object).where(pd.notna(tabela), None)
```

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
import pandas as pd

bronze = pd.read_csv("etl_northwind/bronze/vendas_bruto.csv")
prata = pd.read_csv("etl_northwind/prata/vendas_limpo.csv")

bronze_pronto = bronze.astype(object).where(pd.notna(bronze), None)
prata_pronto = prata.astype(object).where(pd.notna(prata), None)

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor = conexao.cursor()

try:
    cursor.execute("TRUNCATE TABLE analytics.bronze_vendas")
    execute_values(cursor, "INSERT INTO analytics.bronze_vendas VALUES %s",
                   list(bronze_pronto.itertuples(index=False, name=None)))

    cursor.execute("TRUNCATE TABLE analytics.prata_vendas")
    execute_values(cursor, "INSERT INTO analytics.prata_vendas VALUES %s",
                   list(prata_pronto.itertuples(index=False, name=None)))

    conexao.commit()
    print("Bronze e Prata carregadas no banco.")
except Exception as erro:
    conexao.rollback()
    print("Falhou, rollback executado:", erro)

conferencia = pd.read_sql('''
    SELECT 'bronze_vendas' AS tabela, COUNT(*) AS linhas,
           COUNT(*) FILTER (WHERE data_envio IS NULL) AS datas_vazias
    FROM analytics.bronze_vendas
    UNION ALL
    SELECT 'prata_vendas', COUNT(*),
           COUNT(*) FILTER (WHERE data_envio IS NULL)
    FROM analytics.prata_vendas
''', conexao)

print("\nConferência no banco:")
display(conferencia)

conexao.close()

Os números confirmam a tradução correta: a Bronze tem **2155 linhas e 73 datas vazias** — exatamente os vazios que existiam no CSV, agora gravados como `NULL` de verdade. A Prata tem **1784 linhas e nenhuma data vazia**, porque essas 73 linhas foram descartadas pela regra do Dia 2.

`COUNT(*) FILTER (WHERE condição)` é uma forma do PostgreSQL de contar apenas as linhas que atendem a um critério, dentro da mesma consulta que já conta o total. É mais direto do que rodar duas consultas separadas.

### 📖 Antes do código — o medalhão inteiro numa consulta só

Agora que as três camadas vivem no banco, dá para fazer o que motivou este passo: descer de uma camada para a outra com SQL.

A consulta abaixo pega a categoria com maior receita na camada Ouro e vai até a Prata buscar as vendas que formaram aquele número — a prova de origem que um CSV não dá.

A subconsulta entre parênteses no `WHERE` é executada primeiro, e o resultado dela alimenta a consulta de fora. Você viu essa estrutura na Semana 10.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

rastreio = pd.read_sql('''
    SELECT categoria, cliente, produto, quantidade, receita, data_pedido
    FROM analytics.prata_vendas
    WHERE categoria = (
        SELECT categoria
        FROM analytics.ouro_receita_categoria
        ORDER BY receita_total DESC
        LIMIT 1
    )
    ORDER BY receita DESC
    LIMIT 10
''', conexao)

resumo = pd.read_sql('''
    SELECT categoria, receita_total, pedidos
    FROM analytics.ouro_receita_categoria
    ORDER BY receita_total DESC
    LIMIT 1
''', conexao)

conexao.close()

print("Na camada OURO, a categoria líder é:")
display(resumo)

print("\nNa camada PRATA, as 10 maiores vendas que formaram esse número:")
display(rastreio)

Isso é rastreabilidade de verdade: um número agregado, e o caminho de volta até as linhas que o produziram — tudo dentro do banco, com SQL.

**Uma pergunta que talvez tenha surgido:** o pipeline do Passo 18 carrega apenas as três tabelas da camada Ouro, e não a Bronze e a Prata que você acabou de levar para o banco. Isso foi proposital — o pipeline daquele passo mostra o esqueleto completo (extrair, transformar, agregar, carregar, validar, registrar) com o mínimo de código para caber numa leitura só.

Estender o pipeline para carregar também a Bronze e a Prata é o passo natural, e você já tem todas as peças: bastaria criar uma função `carregar_camada(tabela, nome)` que aplica o `.astype(object).where(pd.notna(...), None)` antes do `TRUNCATE` + `execute_values`, e chamá-la para as duas camadas dentro de `executar_pipeline()`. Fica como desafio para quem terminar antes.

## Passo 20 — O painel do seu ETL dentro do notebook

Até aqui, toda vez que você quis conferir uma tabela do banco, a alternativa era abrir o pgAdmin. Trocar de janela o tempo todo atrapalha, e existe um caminho melhor: o próprio notebook pode funcionar como explorador do banco.

São três células, e juntas elas fazem o que você usa do pgAdmin no dia a dia:

| O que você quer | No pgAdmin | Aqui |
|---|---|---|
| Ver quais tabelas existem e o tamanho de cada uma | árvore lateral | Explorador 1 |
| Abrir uma tabela e olhar o conteúdo | botão direito → View Data | Explorador 2 |
| Rodar uma consulta qualquer | Query Tool | Explorador 3 |

### 📖 Antes do código — Explorador 1: o inventário do schema

O código percorre as tabelas do schema `analytics` e monta um inventário com o nome, a quantidade de colunas e a contagem real de linhas de cada uma.

O `for` é necessário aqui por um motivo específico: `COUNT(*)` só funciona numa tabela por vez, então a contagem precisa de uma consulta para cada tabela. O nome da tabela entra na consulta por f-string (`f"SELECT COUNT(*) FROM analytics.{nome}"`), e isso merece um aviso de segurança.

Na Semana 11 você aprendeu a nunca colar valor no comando com formatação de texto, sempre usar `%s`. A regra continua valendo — mas ela vale para **valores**, e o `%s` não funciona para **nome de tabela** (o banco espera um identificador ali, não um dado). Por isso, quando o nome de objeto precisa ser dinâmico, a f-string é a única saída. Ela só é segura aqui porque o nome vem do catálogo do próprio banco, não de algo digitado por um usuário. Se viesse de um formulário, seria uma porta aberta para SQL Injection.

Repare também no `params={"t": nome}` na segunda consulta: essa é a forma nomeada de passar parâmetro no `pd.read_sql()`, equivalente ao `%s` posicional. O marcador no SQL vira `%(t)s`.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

tabelas = pd.read_sql('''
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'analytics' AND table_type = 'BASE TABLE'
    ORDER BY table_name
''', conexao)

inventario = []
for nome in tabelas["table_name"]:
    linhas = pd.read_sql(f"SELECT COUNT(*) AS total FROM analytics.{nome}", conexao)["total"][0]
    colunas = pd.read_sql('''
        SELECT COUNT(*) AS quantas
        FROM information_schema.columns
        WHERE table_schema = 'analytics' AND table_name = %(t)s
    ''', conexao, params={"t": nome})["quantas"][0]
    inventario.append({"tabela": nome, "colunas": colunas, "linhas": linhas})

conexao.close()

print("Inventário do schema analytics:")
display(pd.DataFrame(inventario))

### 📖 Antes do código — Explorador 2: abrindo uma tabela

Esta célula é a que substitui o "View Data" do pgAdmin. A variável `TABELA`, no topo, é o que você troca para olhar outra tabela — os nomes disponíveis são os que apareceram no inventário acima.

`LIMIT 50` no fim da consulta evita trazer uma tabela inteira para a tela sem necessidade. As tabelas da camada Ouro são pequenas, mas o hábito de limitar vale desde já: numa tabela de milhões de linhas, um `SELECT *` sem limite trava o notebook.

Depois do `display()`, o código mostra também os tipos das colunas com `.dtypes` — assim você vê, na mesma célula, o conteúdo e a estrutura.

In [ ]:
import psycopg2
import pandas as pd

TABELA = "ouro_top_clientes"   # troque aqui: ouro_receita_categoria, ouro_receita_mensal, etl_execucao

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

dados = pd.read_sql(f"SELECT * FROM analytics.{TABELA} LIMIT 50", conexao)
conexao.close()

print(f"analytics.{TABELA} — {len(dados)} linhas × {len(dados.columns)} colunas\n")
display(dados)

print("\nTipo de cada coluna:")
display(dados.dtypes.to_frame("tipo"))

### 📖 Antes do código — Explorador 3: sua própria consulta

Esta é a Query Tool do pgAdmin, dentro do notebook. Escreva qualquer SQL na variável `CONSULTA` e rode a célula: o resultado aparece como tabela formatada.

A consulta de exemplo junta duas coisas que você produziu nesta semana: o ranking de categorias e o histórico de execuções do ETL. Troque o conteúdo de `CONSULTA` à vontade — é a célula para experimentar.

`'''` (três aspas) permite escrever um texto de várias linhas, mantendo a indentação do SQL legível. É o mesmo recurso que você vem usando em toda consulta desde o Dia 1.

In [ ]:
import psycopg2
import pandas as pd

CONSULTA = '''
    SELECT categoria,
           receita_total,
           pedidos,
           ROUND(receita_total / pedidos, 2) AS ticket_medio_por_pedido
    FROM analytics.ouro_receita_categoria
    ORDER BY ticket_medio_por_pedido DESC
'''

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

resultado = pd.read_sql(CONSULTA, conexao)
conexao.close()

print("Resultado da consulta:")
display(resultado)

### 👀 E os arquivos CSV? O Rainbow CSV deixa eles legíveis

Para as camadas em disco existe um recurso do VS Code que ajuda muito, e que você provavelmente já tem instalado: a extensão **Rainbow CSV**.

1. Abra qualquer arquivo da pasta `etl_northwind/` (por exemplo `ouro/receita_por_categoria.csv`).
2. O Rainbow CSV pinta **cada coluna de uma cor diferente**, o que já resolve o problema de perder a conta das vírgulas.
3. Para ver o arquivo alinhado como tabela, pressione **Ctrl + Shift + P**, digite `Rainbow CSV: Align` e confirme. As colunas ficam alinhadas verticalmente, como numa planilha.
4. O comando `Rainbow CSV: Shrink` desfaz o alinhamento e devolve o arquivo ao formato original.

Se a extensão não estiver instalada, abra o painel de extensões (**Ctrl + Shift + X**), pesquise por `Rainbow CSV` e instale — é gratuita.

As três células acima resolvem tudo que esta semana precisa, e funcionam em qualquer máquina sem instalar nada. Mas existe um jeito ainda melhor de acompanhar o pipeline: ter o banco aberto **na barra lateral**, ao lado do notebook. É o Passo 21.

### ✏️ Atividade Prática 10 — Sua vez de programar

**Contextualização:** a área de logística da Northwind conseguiu espaço no painel da diretoria e quer um indicador próprio: o desempenho de entrega por país, incluindo o percentual de entregas atrasadas. Como é um indicador novo, ele precisa percorrer o caminho completo — virar tabela no schema `analytics`, ser carregado de forma idempotente e ser validado.

**Comando:** faça as quatro etapas:

1. **Crie a tabela** `analytics.ouro_entregas_pais` com as colunas `pais VARCHAR(50) PRIMARY KEY`, `dias_medio NUMERIC(5,1)`, `entregas INTEGER` e `percentual_atraso NUMERIC(5,2)`. Use `CREATE TABLE IF NOT EXISTS` e lembre do `commit()`.
2. **Agregue** a camada Prata (`pd.read_csv("etl_northwind/prata/vendas_limpo.csv")`) por `pais_cliente`, criando `dias_medio` (média de `dias_entrega`), `entregas` (`"nunique"` sobre `id_pedido`) e `percentual_atraso` (média de `entrega_atrasada` multiplicada por 100). Arredonde e use `reset_index()`.
3. **Carregue** com `TRUNCATE` seguido de `execute_values()`, na mesma transação, com um `commit()` só no fim.
4. **Valide** comparando a contagem de linhas do DataFrame com o `COUNT(*)` da tabela no banco, e imprima se passou ou não.

Ao terminar, confira a tabela nova no pgAdmin (com **Refresh** no schema `analytics`).

In [ ]:
# escreva seu código aqui

---

## Passo 21 — O banco ao lado do notebook: instalando a extensão PostgreSQL

O medalhão inteiro está no banco: seis tabelas no schema `analytics`, da Bronze ao log de execução. Até agora você conferiu tudo por código — e isso continua valendo. Mas no dia a dia de um analista, é muito mais confortável ter o banco aberto **na barra lateral do VS Code**, ao lado do notebook, para olhar uma tabela ou testar uma consulta sem trocar de janela para o pgAdmin.

Quem faz isso é uma **extensão**. Uma extensão é um programa pequeno que acrescenta um recurso novo ao VS Code — do mesmo jeito que a extensão do Python e a do Jupyter são o que permite este notebook rodar aqui. A que você vai instalar se chama **PostgreSQL** e é gratuita e oficial, publicada pela Microsoft.

Este passo é longo porque é feito **uma vez só**, e cada detalhe dele já travou alguém. Faça com calma e na ordem.

### 21.1 — Instalando a extensão

1. No VS Code, abra o painel de extensões: pressione **Ctrl + Shift + X**. Outra forma é clicar no ícone de quatro quadradinhos, na barra de ícones do lado esquerdo da tela.
2. Na caixa de pesquisa, no topo do painel, digite **PostgreSQL**.
3. Vão aparecer várias extensões com nomes parecidos. A certa tem **as três características juntas**:
   - o nome é exatamente **PostgreSQL**;
   - embaixo do nome aparece **Microsoft**, com um selo azul de verificado;
   - ao clicar nela, a página de detalhes mostra o identificador `ms-ossdata.vscode-pgsql`.
4. Clique no botão azul **Install**.
5. Espere terminar. Na primeira vez, a extensão baixa um componente extra, então pode levar um ou dois minutos. Se abrir uma página de boas-vindas, pode fechar a aba.

### 21.2 — Encontrando o ícone da extensão

Instalada, a extensão coloca um **ícone novo na barra de ícones da esquerda** — a mesma barra onde ficam o Explorer (pastas), a Pesquisa (lupa) e as Extensões. O ícone é o **elefante do PostgreSQL**, desenhado só em contorno, sem cor, igual aos outros ícones da barra. Passe o mouse sobre os ícones: quando aparecer o nome **PostgreSQL**, é ele.

**Instalou e o ícone não apareceu?** Quase sempre a janela do VS Code foi aberta antes da instalação e ainda não percebeu a extensão nova. A solução é recarregar a janela — isso **não** fecha o notebook e **não** apaga nada:

1. Pressione **Ctrl + Shift + P**. Abre uma caixa de texto no topo da tela: é a **paleta de comandos**, onde dá para chamar qualquer recurso do VS Code pelo nome.
2. Digite `Reload Window`.
3. Escolha **Developer: Reload Window** e pressione **Enter**.

**Recarregou e continua sem achar?** Dois caminhos:

- A barra pode ter escondido o ícone. Clique com o **botão direito** num espaço vazio da barra de ícones da esquerda. Se **PostgreSQL** aparecer na lista sem marcação, clique para marcar.
- Pressione **Ctrl + Shift + P** e digite `PGSQL`. Todos os comandos da extensão começam com esse prefixo. Se a lista aparecer, a extensão está instalada e funcionando — só o ícone que está escondido.

### 21.3 — Criando a conexão com o northwind

1. Clique no ícone do elefante. Abre o painel **PostgreSQL**, com uma seção chamada **Connections**.
2. Clique em **Add New Connection** (ícone no topo do painel). Se preferir o teclado: **Ctrl + Shift + P** → `PGSQL: Add New Connection`.
3. Um formulário abre no meio da tela. No topo dele está escrito **CONNECT VIA**, com três abas: **Parameters**, **Connection String** e **Browse Azure**. Deixe marcada a primeira, **Parameters**.
4. Preencha campo por campo, na ordem em que eles aparecem:

| Campo no formulário | O que preencher | Por quê |
|---|---|---|
| **SERVER NAME** | `localhost` | "o PostgreSQL instalado nesta mesma máquina" |
| **AUTHENTICATION TYPE** | `Password` | escolha na lista; é o login com usuário e senha |
| **USER NAME** | `postgres` | o mesmo usuário do `psycopg2.connect()` |
| **PASSWORD** | a **sua** senha do PostgreSQL | a mesma que substituiu `SUA_SENHA_AQUI` no notebook |
| **SAVE PASSWORD** | ✅ marcado | sem isso, a extensão pede a senha a cada conexão |
| **DATABASE NAME** | `northwind` | ⚠️ **não deixe vazio** — explicação logo abaixo |
| **CONNECTION NAME** | `Northwind (aula T5)` | é só o apelido que aparece na lista |
| **SERVER GROUP** | `Servers` | deixe o valor que já vem preenchido |

5. O botão **Advanced** guarda configurações extras, como a **porta**. Só mexa nele se, ao instalar o PostgreSQL, você escolheu uma porta diferente da padrão, `5432`.
6. Clique em **Connect**, no final do formulário.

**⚠️ Por que o DATABASE NAME não pode ficar vazio.** O formulário deixa conectar sem esse campo, e é exatamente aí que muita gente se perde. Com ele vazio, a extensão entra no banco `postgres` — um banco administrativo que todo servidor PostgreSQL tem. A conexão **funciona**, a árvore aparece, mas as tabelas da Northwind não estão lá, porque moram em outro banco. Se aconteceu com você, não precisa refazer nada: clique com o **botão direito** na conexão, escolha **Edit Connection**, preencha `northwind` em **DATABASE NAME** e salve.

### 21.4 — Conferindo que a extensão conectou de verdade

A árvore aparecendo já é uma boa pista. Mas existe uma prova que vem do próprio servidor, e ela ensina uma coisa útil sobre o PostgreSQL.

### 📖 Antes do código — `pg_stat_activity`

Todo programa que abre uma conexão com o PostgreSQL ganha uma **sessão** no servidor. A visão `pg_stat_activity` é onde o PostgreSQL lista todas as sessões abertas naquele instante: qual banco cada uma usa, com qual usuário, desde que horas, e qual programa a abriu.

A coluna que resolve a dúvida é `application_name`, o nome que cada programa informa ao se conectar. A extensão do VS Code se identifica como `vscode-pgsql`, e o pgAdmin como `pgAdmin 4`. O `psycopg2` deste notebook não informa nome, então aparece em branco.

O filtro `backend_type = 'client backend'` deixa de fora os processos internos do próprio PostgreSQL, que também aparecem nessa visão mas não são conexões de programas.

`to_char(backend_start, 'DD/MM/YYYY HH24:MI')` é a função do PostgreSQL que transforma uma data em texto no formato que você escolher — aqui, o padrão brasileiro.

O `if`/`elif`/`else` no final lê a tabela e escreve o diagnóstico: extensão desconectada, conectada no banco certo, ou conectada no banco errado. Para isso ele usa `.any()`, que devolve `True` se **pelo menos um** valor de uma coluna de verdadeiro/falso for verdadeiro — aqui, se alguma das sessões da extensão está no banco `northwind`.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

sessoes = pd.read_sql('''
    SELECT application_name AS programa,
           datname AS banco,
           usename AS usuario,
           to_char(backend_start, 'DD/MM/YYYY HH24:MI') AS conectado_desde,
           state AS situacao
    FROM pg_stat_activity
    WHERE backend_type = 'client backend'
    ORDER BY backend_start
''', conexao)

conexao.close()

display(sessoes)

extensao = sessoes[sessoes["programa"] == "vscode-pgsql"]
if len(extensao) == 0:
    print("Nenhuma sessão da extensão do VS Code: ela ainda não conectou. Volte ao passo 21.3.")
elif (extensao["banco"] == "northwind").any():
    print("A extensão do VS Code está conectada ao banco northwind. Tudo certo!")
else:
    print("A extensão está conectada, mas no banco", extensao["banco"].iloc[0],
          "- edite a conexão e preencha DATABASE NAME com northwind.")

Cada linha da tabela é um programa conectado ao seu PostgreSQL neste momento. Se o pgAdmin estiver aberto, ele também aparece. A linha com `programa` em branco é o próprio notebook, que abriu uma conexão só para fazer essa consulta e já fechou.

### 21.5 — Problemas de instalação e conexão, e como resolver

| O que aconteceu | Por que acontece | Como resolver |
|---|---|---|
| Instalei e o ícone não aparece | a janela foi aberta antes da instalação | **Ctrl + Shift + P** → **Developer: Reload Window** |
| Não encontro o ícone | é discreto, ou a barra escondeu | botão direito na barra de ícones → marcar **PostgreSQL**; ou **Ctrl + Shift + P** → `PGSQL` |
| Fechei o painel e não sei abrir de novo | — | clique no ícone do elefante; ou **Ctrl + Shift + P** → **PostgreSQL: Focus on Connections View** |
| Conectou, mas não vejo as tabelas da Northwind | **DATABASE NAME** ficou vazio | botão direito na conexão → **Edit Connection** → **DATABASE NAME** = `northwind` |
| A extensão pede a senha toda vez | **SAVE PASSWORD** desmarcado | botão direito na conexão → **Edit Connection** → marcar **SAVE PASSWORD** |
| Erro de senha | a senha digitada é outra | é a mesma do `psycopg2`; rode a célula de conexão do Passo 0 para confirmar |
| Fechei o VS Code e a árvore não expande | a conexão caiu quando o programa fechou | clique na conexão **Northwind (aula T5)** para reconectar |

---

## Passo 22 — Manuseando a ferramenta: árvore, tabelas e consultas

Conectar é metade. Agora você aprende a **usar**: navegar, olhar colunas, abrir tabelas, escrever consultas, executar, ler o resultado, lidar com erro, salvar e reabrir. É o mesmo tipo de trabalho que você fazia no pgAdmin nas Semanas 08 a 10 — só que com os botões em outros lugares.

### 22.1 — Conhecendo o painel

O painel **PostgreSQL** tem algumas seções. Duas importam para esta semana:

- **Connections** — a árvore do banco. É onde você vai passar a maior parte do tempo.
- **Query History** — o histórico de todas as consultas que você executou pela extensão.

As outras seções (sobre Azure e migrações) servem para bancos na nuvem e não são usadas neste curso.

### 22.2 — Navegando até o seu medalhão

Clique na setinha ao lado de cada item para expandir. O caminho até as tabelas que o seu pipeline criou é:

```
Northwind (aula T5)                  ← a conexão
└── Databases
    └── northwind                    ← o banco
        └── Schemas
            ├── analytics            ← o que o SEU pipeline criou
            │   └── Tables
            │       ├── bronze_vendas
            │       ├── etl_execucao
            │       ├── ouro_receita_categoria
            │       ├── ouro_receita_mensal
            │       ├── ouro_top_clientes
            │       └── prata_vendas
            │           └── Columns  ← as colunas, com o tipo de cada uma
            └── public               ← o sistema de vendas da Northwind
                └── Tables
                    └── clientes, pedidos, itens_pedido, produtos, categorias...
```

**Não apareceu o `analytics`?** A árvore mostra o banco do jeito que ele estava quando você conectou, e **não se atualiza sozinha**. Clique com o **botão direito** em **Schemas** e escolha **Refresh**. Guarde esse gesto: toda vez que o Python criar uma tabela nova, a árvore só vai mostrar depois de um **Refresh**.

### 22.3 — Três gestos na árvore, sem escrever SQL

**Gesto 1 — Ver colunas e tipos.** Expanda `prata_vendas` → **Columns**. Encontre `receita` (tipo `numeric`), `entrega_atrasada` (tipo `boolean`) e `data_envio` (tipo `date`). São exatamente os tipos que você escolheu no `CREATE TABLE` do Passo 19.

**Gesto 2 — Ver o conteúdo de uma tabela.** Clique com o **botão direito** em `ouro_receita_categoria` e escolha **Select Top 1000**. O resultado aparece numa grade, no painel **PostgreSQL Query Results**, na parte de baixo da tela. As 8 categorias aparecem, com as mesmas receitas que o notebook calculou. O nome do comando diz o que ele faz: traz **no máximo** as primeiras 1000 linhas — em `bronze_vendas`, que tem 2155, você veria só 1000.

**Gesto 3 — Ver o comando que criou a tabela.** Clique com o **botão direito** em `bronze_vendas` → **Script as…** → **Script as Create**. Abre uma aba com o `CREATE TABLE` que o banco guardou para essa tabela. Compare com o que você escreveu no Passo 19: as colunas e os tipos são os mesmos, escritos do jeito do PostgreSQL.

> ⚠️ **Um item do menu para NÃO usar: Edit Data.** Ele permite alterar valores direto na grade, como numa planilha. Nas tabelas do `analytics`, qualquer alteração manual é apagada na próxima execução do pipeline, porque a carga faz `TRUNCATE`. Nas tabelas do `public`, a alteração vai direto para o sistema de vendas da Northwind. Nos dois casos, o dado deixa de passar pelo ETL — que é justamente o que garante a qualidade dele.

### 22.4 — O editor de consultas, passo a passo

Esta é a parte que mais se usa, então ela vai devagar.

**Passo A — Abrir um editor já conectado.** Clique com o **botão direito** em `northwind` (dentro de **Databases**) e escolha **New Query**. Abre uma aba nova, vazia, do tipo SQL.

**Passo B — Confirmar que o editor está conectado.** Olhe para o **canto superior direito da aba**. Quando o editor está conectado, aparecem ali botões como **Execute Query** (um triângulo ▶), **Disconnect**, **Change Connection** e **Change Database**. Se, em vez disso, aparecer só o botão **Connect**, o editor ainda não está ligado a banco nenhum: clique em **Connect** (ou pressione **Ctrl + Shift + C** com o cursor dentro do editor) e escolha **Northwind (aula T5)** na lista.

**Passo C — Escrever a consulta.** Digite no editor:

```sql
SELECT categoria, receita_total, pedidos
FROM analytics.ouro_receita_categoria
ORDER BY receita_total DESC;
```

**Passo D — Executar.** Existem quatro formas de executar, e todas fazem a mesma coisa:

| Forma | Como |
|---|---|
| Botão | clique no triângulo ▶ **Execute Query**, no canto superior direito da aba |
| Atalho 1 | com o cursor **dentro do editor**, pressione **Ctrl + Shift + E** |
| Atalho 2 | com o cursor dentro do editor, pressione **Shift + Enter** |
| Menu | botão direito dentro do editor → **Execute Query** |

> ⚠️ **Cuidado com o Ctrl + Shift + E.** No VS Code, esse mesmo atalho abre o Explorer de arquivos. Ele só executa a consulta quando o cursor está **dentro do editor SQL**. Se o Explorer abriu, clique dentro do texto da consulta e tente de novo — ou use o **Shift + Enter**, que não tem esse conflito.

**Passo E — Ler o resultado.** O resultado aparece no painel **PostgreSQL Query Results**, na parte de baixo da tela, em forma de grade. Se você fechou esse painel sem querer, clique no botão **Reveal Query Result**, no topo da aba da consulta, para trazê-lo de volta.

**Passo F — Errar de propósito, para conhecer a mensagem.** Troque a consulta por esta, com o nome da tabela no plural, e execute:

```sql
SELECT * FROM analytics.ouro_receita_categorias;
```

O painel de resultados mostra um erro dizendo que a relação `analytics.ouro_receita_categorias` **não existe**. "Relação" é como o PostgreSQL chama tabelas e views nas mensagens de erro. Quando essa mensagem aparecer, confira a grafia do nome — é quase sempre um erro de digitação.

**Passo G — O erro mais comum de todos: esquecer o schema.** Execute agora:

```sql
SELECT * FROM ouro_receita_categoria;
```

Também dá erro de relação que não existe — mesmo com o nome escrito certo. O motivo: quando você não informa o schema, o PostgreSQL procura a tabela no schema `public`, e ela está no `analytics`. Por isso, em toda consulta ao seu medalhão, escreva o nome completo: `analytics.nome_da_tabela`. As tabelas do sistema de vendas (`pedidos`, `clientes`...) funcionam sem o prefixo justamente porque moram no `public`.

**Passo H — Várias consultas no mesmo editor.** Um editor pode guardar várias consultas, cada uma terminando com ponto e vírgula. Digite as duas abaixo, uma embaixo da outra:

```sql
SELECT COUNT(*) AS linhas_bronze FROM analytics.bronze_vendas;

SELECT COUNT(*) AS linhas_prata FROM analytics.prata_vendas;
```

Para executar **só uma delas**, clique em qualquer lugar dentro da consulta que você quer e use **Execute Current Statement**: pressione **Ctrl + Shift + Enter**. Executa apenas a instrução onde o cursor está. É o jeito mais prático de ir testando consultas uma a uma, sem apagar as anteriores.

**Passo I — Salvar as consultas num arquivo.** Pressione **Ctrl + S**, escolha a pasta onde está este notebook e salve como `consultas_medalhao.sql`. O arquivo aparece no Explorer. Isso é útil de verdade: a consulta que você testou vira um arquivo que pode ser reaberto na aula seguinte, enviado para um colega ou colado no código Python.

**Passo J — Reabrir um arquivo `.sql` depois.** Dê dois cliques no `consultas_medalhao.sql` no Explorer. Repare no canto superior direito: aparece o botão **Connect**, porque um arquivo reaberto **não guarda a conexão**. Clique em **Connect** (ou **Ctrl + Shift + C**), escolha **Northwind (aula T5)**, e as consultas voltam a executar.

**Passo K — Recuperar uma consulta antiga pelo histórico.** Abra a seção **Query History** no painel PostgreSQL. Cada consulta executada aparece ali. Com o **botão direito** num item: **Open Query** abre a consulta num editor, **Run Query** executa de novo, **Copy Query** copia o texto.

**Passo L — O editor abriu no banco errado.** Se um editor ficou conectado ao banco `postgres` em vez do `northwind`, não precisa abrir outro: clique no botão **Change Database**, no topo da aba, e escolha `northwind`.

### 22.5 — Roteiro guiado: investigando o seu medalhão com consultas

Hora de usar a ferramenta como um analista usaria: fazendo perguntas ao dado. Abra um editor com **New Query** em `northwind` e execute as consultas abaixo, **uma de cada vez** (com **Ctrl + Shift + Enter**). Depois de cada uma, confira se o resultado bate com o esperado.

**Consulta 1 — Quantas linhas tem cada camada?**

```sql
SELECT 'bronze' AS camada, COUNT(*) AS linhas FROM analytics.bronze_vendas
UNION ALL
SELECT 'prata', COUNT(*) FROM analytics.prata_vendas
UNION ALL
SELECT 'ouro (categorias)', COUNT(*) FROM analytics.ouro_receita_categoria;
```

✅ Esperado: **2155**, **1784** e **8**. O encolhimento do pipeline, direto do banco.

**Consulta 2 — Para onde foram as linhas que sumiram entre a Bronze e a Prata?**

```sql
SELECT 'sem data de envio' AS motivo, COUNT(*) AS linhas
FROM analytics.bronze_vendas
WHERE data_envio IS NULL
UNION ALL
SELECT 'produto descontinuado', COUNT(*)
FROM analytics.bronze_vendas
WHERE data_envio IS NOT NULL AND descontinuado = 1;
```

✅ Esperado: **73** e **298**. Somando, **371** — exatamente a diferença entre 2155 e 1784. Cada linha descartada tem um motivo de negócio, e dá para provar com uma consulta. Repare no `IS NULL`: em SQL, vazio não se compara com `=`, e sim com `IS NULL` ou `IS NOT NULL`.

**Consulta 3 — A camada Ouro está coerente com a Prata?**

```sql
SELECT (SELECT SUM(receita) FROM analytics.prata_vendas) AS receita_na_prata,
       (SELECT SUM(receita_total) FROM analytics.ouro_receita_categoria) AS receita_na_ouro;
```

✅ Esperado: os dois valores iguais, **1032444.79**. Se algum dia forem diferentes, a agregação perdeu ou duplicou dinheiro no caminho.

**Consulta 4 — Em qual categoria as entregas mais atrasam?**

```sql
SELECT categoria,
       COUNT(*) AS itens,
       COUNT(*) FILTER (WHERE entrega_atrasada) AS itens_atrasados,
       ROUND(100.0 * COUNT(*) FILTER (WHERE entrega_atrasada) / COUNT(*), 1) AS percentual_atraso
FROM analytics.prata_vendas
GROUP BY categoria
ORDER BY percentual_atraso DESC;
```

✅ Esperado: 8 linhas, uma por categoria. Anote qual categoria lidera — a célula de código logo abaixo roda a mesma consulta pelo Python, e o resultado precisa ser igual. O `100.0` com casa decimal é proposital: com `100` inteiro, o PostgreSQL faria divisão inteira e arredondaria o percentual para baixo.

**Consulta 5 — O que o pipeline registrou?**

```sql
SELECT * FROM analytics.etl_execucao ORDER BY id DESC;
```

✅ Esperado: uma linha por carga feita, com data, hora e status. É o log do Passo 17, lido pela ferramenta.

Ao terminar, salve o editor como `consultas_medalhao.sql` (**Ctrl + S**) — é o seu primeiro arquivo de consultas de analista.

### 📖 Antes do código — a mesma investigação pelo notebook

A célula abaixo executa as consultas 1 a 4 pelo Python. O objetivo é fechar o ciclo: **a mesma pergunta, feita em dois lugares, precisa dar a mesma resposta**. Se a ferramenta e o notebook discordarem, um dos dois está consultando outra coisa — outro banco, outro schema, ou uma versão antiga da tabela antes de um **Refresh**.

Nenhuma função nova: são `pd.read_sql()` e `display()` com as consultas que você acabou de executar na extensão.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

camadas = pd.read_sql('''
    SELECT 'bronze' AS camada, COUNT(*) AS linhas FROM analytics.bronze_vendas
    UNION ALL
    SELECT 'prata', COUNT(*) FROM analytics.prata_vendas
    UNION ALL
    SELECT 'ouro (categorias)', COUNT(*) FROM analytics.ouro_receita_categoria
''', conexao)

descartes = pd.read_sql('''
    SELECT 'sem data de envio' AS motivo, COUNT(*) AS linhas
    FROM analytics.bronze_vendas WHERE data_envio IS NULL
    UNION ALL
    SELECT 'produto descontinuado', COUNT(*)
    FROM analytics.bronze_vendas WHERE data_envio IS NOT NULL AND descontinuado = 1
''', conexao)

coerencia = pd.read_sql('''
    SELECT (SELECT SUM(receita) FROM analytics.prata_vendas) AS receita_na_prata,
           (SELECT SUM(receita_total) FROM analytics.ouro_receita_categoria) AS receita_na_ouro
''', conexao)

atrasos = pd.read_sql('''
    SELECT categoria,
           COUNT(*) AS itens,
           COUNT(*) FILTER (WHERE entrega_atrasada) AS itens_atrasados,
           ROUND(100.0 * COUNT(*) FILTER (WHERE entrega_atrasada) / COUNT(*), 1) AS percentual_atraso
    FROM analytics.prata_vendas
    GROUP BY categoria
    ORDER BY percentual_atraso DESC
''', conexao)

conexao.close()

print("Consulta 1 - linhas por camada:")
display(camadas)

print("\nConsulta 2 - motivos de descarte:")
display(descartes)
print("Total descartado:", descartes["linhas"].sum(),
      "| Diferença Bronze - Prata:", camadas["linhas"][0] - camadas["linhas"][1])

print("\nConsulta 3 - coerência Prata x Ouro:")
display(coerencia)

print("\nConsulta 4 - atraso por categoria:")
display(atrasos)

### 22.6 — Quando usar a ferramenta, e quando usar o notebook

As duas coisas convivem, e cada uma serve para um momento:

| Situação | Use |
|---|---|
| Conferir rápido se uma tabela foi criada, e com quais colunas | a **árvore** da extensão |
| Olhar o conteúdo de uma tabela | **Select Top 1000** na extensão |
| Testar uma consulta antes de colocar no código | o **editor de consultas** da extensão |
| Construir, limpar, carregar e validar os dados | o **notebook** (Python) |
| Qualquer coisa que precisa rodar de novo amanhã, igual | o **notebook** — o que você clica na ferramenta não fica registrado como processo |

A última linha é a mais importante. A ferramenta é para **olhar e investigar**; o pipeline é para **fazer**. Uma consulta testada na extensão só vira parte do ETL quando entra no código.

### 22.7 — Problemas no uso das consultas, e como resolver

| O que aconteceu | Por que acontece | Como resolver |
|---|---|---|
| Apertei **Ctrl + Shift + E** e abriu o Explorer | o cursor estava fora do editor SQL | clique dentro da consulta e aperte de novo; ou use **Shift + Enter** |
| Só aparece o botão **Connect** no topo do editor | o editor não está ligado a um banco (comum em arquivo `.sql` reaberto) | clique em **Connect** (ou **Ctrl + Shift + C**) → **Northwind (aula T5)** |
| Erro "relação ... não existe", com o nome certo | faltou o schema na frente | escreva `analytics.nome_da_tabela` |
| Erro "relação ... não existe" em tabela que acabou de ser criada | o editor está no banco `postgres` | botão **Change Database** → `northwind` |
| A tabela nova não aparece na árvore | a árvore não se atualiza sozinha | botão direito em **Schemas** (ou **Tables**) → **Refresh** |
| O painel de resultados sumiu | foi fechado | botão **Reveal Query Result**, no topo da aba da consulta |
| Queria rodar uma consulta e rodaram todas | foi usado o **Execute Query**, que executa o editor inteiro | cursor dentro da consulta → **Ctrl + Shift + Enter** |
| A consulta não termina nunca | consulta pesada ou travada | botão **Cancel Query**, no topo da aba |

---

## Passo 23 — A entrega do analista: o relatório da camada Ouro

Chegou o momento para o qual o pipeline inteiro existe. A diretoria da Northwind não vai abrir notebook, CSV nem banco — ela quer um **relatório**: gráficos que respondem perguntas de negócio, e conclusões escritas em português claro.

Imagine que você é a pessoa de dados da Northwind e esta é a sua entrega. Ela tem três partes:

1. **Quatro gráficos**, cada um respondendo uma pergunta da diretoria.
2. **Um painel executivo**: os quatro gráficos numa imagem só, salva em arquivo para anexar num e-mail ou numa apresentação.
3. **As conclusões**, geradas a partir dos números — não digitadas à mão.

**Uma regra que vale para toda esta entrega:** os dados vêm das tabelas `analytics.ouro_*` do **banco**, e não dos CSVs. Essa escolha é deliberada. Ler do banco prova que a carga funcionou, e é exatamente de onde um painel de verdade (Power BI, Metabase) leria.

### 📖 Antes do código — gráfico com `fig` e `ax`

Até agora você desenhou gráficos com `plt.barh()` e `plt.plot()`, chamando tudo direto pelo `plt`. Para relatórios, existe uma forma mais organizada, e ela vai ser necessária para montar o painel com quatro gráficos:

`fig, ax = plt.subplots(figsize=(largura, altura))` cria duas coisas de uma vez. A `fig` (figura) é a **folha** inteira. O `ax` (eixo) é a **área de desenho** onde o gráfico fica. A partir daí, você desenha chamando os métodos do `ax`: `ax.barh()`, `ax.set_title()`, `ax.set_xlabel()`. Os nomes quase não mudam — `plt.title()` vira `ax.set_title()`, `plt.xlabel()` vira `ax.set_xlabel()`.

`ax.bar_label(barras, labels=lista)` escreve um texto na ponta de cada barra. As `barras` são o que o `ax.barh()` devolve, e a `lista` tem um texto por barra. O `padding=3` afasta o texto 3 pontos da barra, para não encostar.

`f"{valor:,.0f}"` formata um número com separador de milhar e sem casas decimais: `230951.17` vira `230,951`. O padrão americano usa vírgula no milhar; o `.replace(",", ".")` troca pela convenção brasileira, e o resultado fica `230.951`.

`zip(lista1, lista2)` percorre duas listas ao mesmo tempo, devolvendo um par por volta. É assim que cada rótulo junta o valor e o percentual da **mesma** categoria.

`ax.set_xlim(0, limite)` define até onde vai o eixo horizontal. O código estica o eixo 35% além da maior receita, para os rótulos caberem dentro do gráfico em vez de passarem da borda.

`ax.invert_yaxis()` é o mesmo `invert_yaxis()` do Passo 11: coloca o maior valor no topo.

A coluna `participacao` é calculada no próprio Pandas: a receita da categoria dividida pela receita total, vezes 100. É ela que responde "quanto da empresa depende desta categoria".

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

categorias = pd.read_sql(
    "SELECT * FROM analytics.ouro_receita_categoria ORDER BY receita_total DESC", conexao
)

conexao.close()

receita_total = categorias["receita_total"].sum()
categorias["participacao"] = (100 * categorias["receita_total"] / receita_total).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
barras = ax.barh(categorias["categoria"], categorias["receita_total"], color="#78C043")
ax.invert_yaxis()
ax.bar_label(
    barras,
    labels=[f"R$ {valor:,.0f}".replace(",", ".") + f"  ({parte}%)"
            for valor, parte in zip(categorias["receita_total"], categorias["participacao"])],
    padding=3,
)
ax.set_title("Pergunta 1 - Onde está a receita da Northwind?")
ax.set_xlabel("Receita total (R$)")
ax.set_xlim(0, categorias["receita_total"].max() * 1.35)
fig.tight_layout()
plt.show()

display(categorias[["categoria", "receita_total", "participacao", "pedidos"]])

Duas categorias, **Dairy Products** e **Beverages**, somam juntas quase metade da receita — cerca de 22% cada. Na outra ponta, **Meat/Poultry** responde por 2%. Para a diretoria, a leitura é direta: uma falha de fornecimento em laticínios ou bebidas afeta a empresa inteira; em carnes, quase não aparece no resultado.

### 📖 Antes do código — a série mensal, sem cair na armadilha do mês incompleto

No Passo 11 você viu que o último mês despenca porque o banco termina no começo dele. Num relatório para a diretoria, isso não pode ser só comentado: o gráfico precisa **mostrar** que aquele ponto é diferente, e a regra precisa ser **calculada**, não decidida no olho.

A regra: descobrir a data da última venda na camada Prata e verificar se ela cai antes do último dia daquele mês.

`pd.Timestamp(data)` transforma a data que veio do banco num objeto de data do Pandas, que tem recursos extras. Dois deles resolvem o problema: `.day` devolve o dia do mês (4, por exemplo) e `.days_in_month` devolve quantos dias aquele mês tem (31, no caso de maio). Se o dia da última venda é menor que o total de dias do mês, o mês está incompleto.

`tabela.loc[condição, "coluna"] = valor` altera a coluna só nas linhas em que a condição é verdadeira — aqui, marca `completo = False` apenas no mês da última venda.

O gráfico desenha os meses completos com `ax.plot()`, e o mês incompleto separado, com `ax.scatter()` — que desenha **pontos soltos**, sem ligar uma linha. O `label=` de cada um dá o nome que aparece na legenda, e `ax.legend()` desenha a legenda.

`ax.tick_params(axis="x", rotation=45)` gira os rótulos do eixo horizontal, igual ao `plt.xticks(rotation=45)` do Passo 11. `ax.grid()` é o mesmo `plt.grid()` do Passo 11, e `ax.set_ylabel("texto")` é o `plt.ylabel()`, agora chamados pelo `ax`.

No final, `mensal.tail()` mostra as **últimas** 5 linhas da tabela — o contrário do `head()`, que mostra as primeiras. Aqui ele é a escolha certa, porque o mês incompleto é justamente o último da série.

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

mensal = pd.read_sql("SELECT * FROM analytics.ouro_receita_mensal ORDER BY mes", conexao)
ultima_venda = pd.read_sql(
    "SELECT MAX(data_pedido) AS ultima FROM analytics.prata_vendas", conexao
)["ultima"][0]
ultima_venda = pd.Timestamp(ultima_venda)
mes_da_ultima_venda = ultima_venda.strftime("%Y-%m")
mes_esta_incompleto = ultima_venda.day < ultima_venda.days_in_month

conexao.close()

mensal["completo"] = True
if mes_esta_incompleto:
    mensal.loc[mensal["mes"] == mes_da_ultima_venda, "completo"] = False

completos = mensal[mensal["completo"]]
incompletos = mensal[~mensal["completo"]]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(completos["mes"], completos["receita_total"], marker="o", color="#78C043", label="mês completo")
ax.scatter(incompletos["mes"], incompletos["receita_total"], color="orange", s=120, zorder=3,
           label="mês incompleto (não comparar)")
ax.set_title("Pergunta 2 - Como a receita evoluiu mês a mês?")
ax.set_ylabel("Receita (R$)")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

print("Última venda registrada na camada Prata:", ultima_venda.strftime("%d/%m/%Y"))
print("O mês", mes_da_ultima_venda, "está incompleto?", mes_esta_incompleto)
display(mensal.tail())

O ponto laranja no final avisa, sem precisar de nenhuma explicação verbal, que aquele mês não serve para comparação. `~` antes de uma condição inverte o `True` e o `False` — é o "não" do Pandas, e foi assim que `incompletos` pegou as linhas que **não** são completas. O `zorder=3` coloca os pontos na frente da grade.

A série mostra crescimento consistente de 1996 para 1998 nos meses completos. Um cuidado que a diretoria precisa ouvir: **não compare o total de 1996 com o de 1997**. O banco começa em julho de 1996 e termina em maio de 1998, então os dois anos das pontas têm menos meses que 1997.

### 📖 Antes do código — quem sustenta a receita

A terceira pergunta é sobre dependência de clientes: se os maiores clientes forem poucos e muito grandes, perder um deles é um problema sério.

`cumsum()` calcula a **soma acumulada**: cada linha passa a valer ela mesma mais todas as anteriores. Aplicada à participação dos clientes ordenados do maior para o menor, ela responde "os 3 maiores clientes somados representam quanto?", "os 5 maiores?", e assim por diante.

O gráfico usa `ax.barh()` para os clientes e uma segunda escala só para o acumulado: `ax.twiny()` cria um **segundo eixo horizontal** no topo do mesmo gráfico, permitindo desenhar a linha do acumulado (de 0 a 100%) por cima das barras de receita (em reais), cada uma com a sua escala.

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

clientes = pd.read_sql("SELECT * FROM analytics.ouro_top_clientes ORDER BY receita_total DESC", conexao)
categorias = pd.read_sql(
    "SELECT * FROM analytics.ouro_receita_categoria ORDER BY receita_total DESC", conexao
)

conexao.close()

receita_total = categorias["receita_total"].sum()
clientes["participacao"] = (100 * clientes["receita_total"] / receita_total).round(1)
clientes["participacao_acumulada"] = clientes["participacao"].cumsum().round(1)

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(clientes["cliente"], clientes["receita_total"], color="#78C043")
ax.invert_yaxis()
ax.set_xlabel("Receita (R$)")
ax.set_title("Pergunta 3 - Quem sustenta a receita? (10 maiores clientes)")

eixo_acumulado = ax.twiny()
eixo_acumulado.plot(clientes["participacao_acumulada"], clientes["cliente"], color="black", marker="o")
eixo_acumulado.set_xlim(0, 100)
eixo_acumulado.set_xlabel("Participação acumulada na receita total (%)")

fig.tight_layout()
plt.show()

display(clientes[["cliente", "pais", "receita_total", "participacao", "participacao_acumulada"]])

A linha preta sobe a cada cliente somado. O número que importa está no último ponto: os **10 maiores clientes**, de um total de 89 que compraram, respondem por perto de **45% da receita**. Só o primeiro, o QUICK-Stop, representa mais de 9%.

### 📖 Antes do código — quanto vale cada pedido

A receita total mistura duas coisas: quantos pedidos acontecem e quanto cada pedido vale. A quarta pergunta separa a segunda: o **ticket médio por pedido**, que é a receita da categoria dividida pelo número de pedidos que tiveram produtos dela.

`ax.bar()` desenha barras **verticais**. Com 8 categorias, cabem bem na horizontal da folha.

`ax.axhline(valor)` desenha uma **linha horizontal** atravessando o gráfico inteiro na altura indicada. Ela marca o ticket médio geral, e assim fica visível, sem fazer conta, quais categorias estão acima ou abaixo da média. O `linestyle="--"` desenha a linha tracejada.

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

categorias = pd.read_sql(
    "SELECT * FROM analytics.ouro_receita_categoria ORDER BY receita_total DESC", conexao
)

conexao.close()

categorias["ticket_medio"] = (categorias["receita_total"] / categorias["pedidos"]).round(2)
categorias = categorias.sort_values("ticket_medio", ascending=False)
media_geral = categorias["ticket_medio"].mean()
media_formatada = f"R$ {media_geral:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(categorias["categoria"], categorias["ticket_medio"], color="#78C043")
ax.axhline(media_geral, color="black", linestyle="--", label=f"média geral: {media_formatada}")
ax.set_title("Pergunta 4 - Em qual categoria cada pedido vale mais?")
ax.set_ylabel("Ticket médio por pedido (R$)")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
plt.show()

display(categorias[["categoria", "receita_total", "pedidos", "ticket_medio"]])

Aqui aparece algo que o gráfico da Pergunta 1 escondia. **Beverages** tem receita quase igual à de Dairy Products, mas com bem menos pedidos — cada pedido de bebidas vale, em média, mais do que qualquer outro. **Produce** vende pouco no total, mas também tem ticket alto. Olhar só a receita total levaria a diretoria a tratar essas categorias como iguais, quando elas se comportam de jeitos diferentes.

A formatação da média usa um truque de troca em três passos. O Python escreve `1,234.56` (padrão americano). Para chegar em `1.234,56`, a vírgula vira um `X` temporário, o ponto vira vírgula, e o `X` vira ponto. Sem o `X` no meio, a segunda troca desfaria a primeira.

### 📖 Antes do código — o painel executivo

O painel junta as quatro respostas numa imagem só.

`fig, ax = plt.subplots(2, 2, figsize=(16, 11))` cria uma folha dividida em **2 linhas e 2 colunas** de gráficos. Agora o `ax` não é uma área só: é uma grade, e cada área é acessada pela posição — `ax[0, 0]` é o canto superior esquerdo, `ax[0, 1]` o superior direito, `ax[1, 0]` o inferior esquerdo, `ax[1, 1]` o inferior direito. Cada um recebe os mesmos comandos que você acabou de usar.

`fig.suptitle("texto")` escreve um título geral acima dos quatro gráficos.

`fig.savefig(caminho, dpi=150, bbox_inches="tight")` salva a folha inteira como imagem. O `dpi` (pontos por polegada) define a nitidez: 150 fica bom tanto na tela quanto impresso. O `bbox_inches="tight"` corta as margens brancas sobrando em volta.

Depois de salvar, o código confere que o arquivo existe com `os.path.exists()` e mostra o tamanho com `os.path.getsize()` — a mesma regra de sempre: não basta mandar salvar, é preciso confirmar que salvou.

In [ ]:
import os
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

categorias = pd.read_sql(
    "SELECT * FROM analytics.ouro_receita_categoria ORDER BY receita_total DESC", conexao
)
mensal = pd.read_sql("SELECT * FROM analytics.ouro_receita_mensal ORDER BY mes", conexao)
clientes = pd.read_sql("SELECT * FROM analytics.ouro_top_clientes ORDER BY receita_total DESC", conexao)
ultima_venda = pd.read_sql(
    "SELECT MAX(data_pedido) AS ultima FROM analytics.prata_vendas", conexao
)["ultima"][0]
ultima_venda = pd.Timestamp(ultima_venda)
mes_da_ultima_venda = ultima_venda.strftime("%Y-%m")
mes_esta_incompleto = ultima_venda.day < ultima_venda.days_in_month

conexao.close()

receita_total = categorias["receita_total"].sum()
categorias["participacao"] = (100 * categorias["receita_total"] / receita_total).round(1)
categorias["ticket_medio"] = (categorias["receita_total"] / categorias["pedidos"]).round(2)

mensal["completo"] = True
if mes_esta_incompleto:
    mensal.loc[mensal["mes"] == mes_da_ultima_venda, "completo"] = False
completos = mensal[mensal["completo"]]
incompletos = mensal[~mensal["completo"]]

fig, ax = plt.subplots(2, 2, figsize=(16, 11))

barras = ax[0, 0].barh(categorias["categoria"], categorias["receita_total"], color="#78C043")
ax[0, 0].invert_yaxis()
ax[0, 0].bar_label(barras, labels=[f"{parte}%" for parte in categorias["participacao"]], padding=3)
ax[0, 0].set_xlim(0, categorias["receita_total"].max() * 1.15)
ax[0, 0].set_title("1. Receita por categoria")
ax[0, 0].set_xlabel("Receita (R$)")

ax[0, 1].plot(completos["mes"], completos["receita_total"], marker="o", color="#78C043", label="mês completo")
ax[0, 1].scatter(incompletos["mes"], incompletos["receita_total"], color="orange", s=100, zorder=3, label="mês incompleto")
ax[0, 1].set_title("2. Evolução mensal da receita")
ax[0, 1].tick_params(axis="x", rotation=90)
ax[0, 1].legend()

ax[1, 0].barh(clientes["cliente"], clientes["receita_total"], color="#78C043")
ax[1, 0].invert_yaxis()
ax[1, 0].set_title("3. Os 10 maiores clientes")
ax[1, 0].set_xlabel("Receita (R$)")

ordem_ticket = categorias.sort_values("ticket_medio", ascending=False)
ax[1, 1].bar(ordem_ticket["categoria"], ordem_ticket["ticket_medio"], color="#78C043")
ax[1, 1].axhline(ordem_ticket["ticket_medio"].mean(), color="black", linestyle="--")
ax[1, 1].set_title("4. Ticket médio por pedido")
ax[1, 1].tick_params(axis="x", rotation=45)

fig.suptitle("Northwind - Painel executivo de vendas (camada Ouro)", fontsize=16)
fig.tight_layout()

os.makedirs("etl_northwind/ouro", exist_ok=True)
fig.savefig("etl_northwind/ouro/painel_executivo.png", dpi=150, bbox_inches="tight")
plt.show()

caminho = "etl_northwind/ouro/painel_executivo.png"
print("Painel salvo?", os.path.exists(caminho))
print("Tamanho do arquivo:", round(os.path.getsize(caminho) / 1024), "KB")

### 👀 Veja o painel no VS Code

1. Abra o Explorer (ícone de pastas na barra da esquerda).
2. Entre em `etl_northwind` → `ouro`.
3. Clique em `painel_executivo.png`. O VS Code abre a imagem numa aba, e dá para aproximar com a rolagem do mouse.

Esse arquivo é a peça que sai do seu computador: vai anexado no e-mail, colado na apresentação, impresso para a reunião.

### 📖 Antes do código — as conclusões escritas pelo próprio código

Um relatório termina com conclusões. O erro comum é escrever os números à mão — e aí, quando o pipeline roda de novo com dados novos, o texto fica desatualizado e ninguém percebe.

A saída é o próprio código montar as frases a partir dos dados. Cada número vem de uma conta feita na hora.

`idxmax()` devolve a **posição** (o índice) do maior valor de uma coluna. Com `.loc[posição]`, você pega a linha inteira onde está esse maior valor — é assim que o código descobre não só qual foi a maior receita mensal, mas **qual mês** foi.

`.iloc[0]` pega a primeira linha de um DataFrame pela posição, e `.iloc[-1]` pega a última. Como `categorias` e `clientes` já vêm ordenados do maior para o menor, `.iloc[0]` é sempre o líder.

`"\n".join(lista)` junta os textos de uma lista num texto só, com uma quebra de linha entre cada um.

`open(caminho, "w", encoding="utf-8")` abre um arquivo para escrita (`"w"` de *write*). O `encoding="utf-8"` é obrigatório no Windows: sem ele, os acentos das conclusões podem sair trocados por símbolos estranhos. O `with` garante que o arquivo é fechado no final, e `arquivo.write(texto)` grava o texto dentro dele.

Para confirmar, o arquivo é aberto de novo, agora sem o `"w"` (só para leitura), e `arquivo.read()` lê o conteúdo inteiro como texto.

In [ ]:
import psycopg2
import pandas as pd

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)

categorias = pd.read_sql(
    "SELECT * FROM analytics.ouro_receita_categoria ORDER BY receita_total DESC", conexao
)
mensal = pd.read_sql("SELECT * FROM analytics.ouro_receita_mensal ORDER BY mes", conexao)
clientes = pd.read_sql("SELECT * FROM analytics.ouro_top_clientes ORDER BY receita_total DESC", conexao)
ultima_venda = pd.read_sql(
    "SELECT MAX(data_pedido) AS ultima FROM analytics.prata_vendas", conexao
)["ultima"][0]
ultima_venda = pd.Timestamp(ultima_venda)
mes_da_ultima_venda = ultima_venda.strftime("%Y-%m")
mes_esta_incompleto = ultima_venda.day < ultima_venda.days_in_month

conexao.close()

def reais(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

receita_total = categorias["receita_total"].sum()
lider = categorias.iloc[0]
lanterna = categorias.iloc[-1]
duas_maiores = 100 * categorias["receita_total"].head(2).sum() / receita_total

categorias["ticket_medio"] = categorias["receita_total"] / categorias["pedidos"]
maior_ticket = categorias.loc[categorias["ticket_medio"].idxmax()]

completos = mensal[mensal["mes"] != mes_da_ultima_venda] if mes_esta_incompleto else mensal
melhor_mes = completos.loc[completos["receita_total"].idxmax()]

maior_cliente = clientes.iloc[0]
peso_top10 = 100 * clientes["receita_total"].sum() / receita_total

if mes_esta_incompleto:
    aviso_mes = (f"8. Atenção: o mês {mes_da_ultima_venda} está incompleto "
                 f"(última venda em {ultima_venda.strftime('%d/%m/%Y')}) e não deve ser comparado.")
else:
    aviso_mes = "8. Todos os meses do período estão completos."

conclusoes = [
    "RELATÓRIO DE VENDAS - NORTHWIND (camada Ouro)",
    "",
    f"1. A receita total analisada é de {reais(receita_total)}.",
    f"2. {lider['categoria']} lidera com {reais(lider['receita_total'])}; as duas maiores categorias somam {duas_maiores:.1f}% da receita.",
    f"3. {lanterna['categoria']} é a categoria de menor receita, com {reais(lanterna['receita_total'])}.",
    f"4. O maior ticket médio por pedido é de {maior_ticket['categoria']}: {reais(maior_ticket['ticket_medio'])}.",
    f"5. O melhor mês completo foi {melhor_mes['mes']}, com {reais(melhor_mes['receita_total'])}.",
    f"6. O maior cliente é {maior_cliente['cliente']} ({maior_cliente['pais']}), com {reais(maior_cliente['receita_total'])}.",
    f"7. Os 10 maiores clientes concentram {peso_top10:.1f}% da receita - risco de dependência a acompanhar.",
    aviso_mes,
]

texto = "\n".join(conclusoes)

with open("etl_northwind/ouro/conclusoes.txt", "w", encoding="utf-8") as arquivo:
    arquivo.write(texto)

print("Conclusões gravadas. Confirmando pela releitura do arquivo:\n")
with open("etl_northwind/ouro/conclusoes.txt", encoding="utf-8") as arquivo:
    print(arquivo.read())

Repare que **nenhum número desse texto foi digitado**. Se amanhã o sistema da Northwind receber mil pedidos novos e o pipeline rodar de novo, as oito frases saem atualizadas sozinhas — inclusive o aviso do mês incompleto, que depende da data da última venda.

A função `reais(valor)` foi criada dentro desta célula porque a mesma formatação de dinheiro aparece várias vezes. É o uso clássico de função que você aprendeu na Semana 04: escrever uma vez, usar várias.

Abra o `conclusoes.txt` no Explorer (`etl_northwind/ouro/`) e veja o arquivo final. A entrega do analista está completa:

```
etl_northwind/ouro/
├── receita_por_categoria.csv
├── receita_mensal.csv
├── top_clientes.csv
├── painel_executivo.png     ← os quatro gráficos
└── conclusoes.txt           ← as conclusões, geradas pelo código
```

### ✏️ Atividade Prática 11 — Sua vez de programar

**Contextualização:** a diretoria comercial gostou do painel e fez uma pergunta que ele ainda não responde: o valor de cada pedido está subindo ou caindo ao longo do tempo? Se a receita cresce só porque entram mais pedidos, mas cada pedido vale cada vez menos, a empresa está trabalhando mais para ganhar o mesmo — e isso muda a estratégia de vendas.

**Comando:** conecte no banco `northwind` e leia `analytics.ouro_receita_mensal` com `pd.read_sql()`. Leia também a data da última venda com `SELECT MAX(data_pedido) AS ultima FROM analytics.prata_vendas`, e use a mesma regra do exemplo (`pd.Timestamp`, `.day` e `.days_in_month`) para **remover** o mês incompleto. Feche a conexão. Crie a coluna `ticket_medio` dividindo `receita_total` por `pedidos`. Com `fig, ax = plt.subplots()`, desenhe a linha do `ticket_medio` mês a mês com `ax.plot()`, uma linha tracejada com a média geral usando `ax.axhline()`, título e rótulo no eixo vertical. Salve o gráfico em `etl_northwind/ouro/ticket_mensal.png` com `fig.savefig()` e confirme com `os.path.exists()`. Por fim, use `print()` com f-string para escrever uma frase dizendo qual foi o mês de maior ticket médio (use `idxmax()`) e o valor.

In [ ]:
# escreva seu código aqui

---

## 🚀 Desafio de Squad — Sexta-feira

Agora em equipe. Cada squad recebe uma consultoria fictícia diferente, mas **todos trabalham sobre o mesmo banco `northwind`** — em todos os casos, a Northwind é o sistema transacional que alimenta a análise.

A entrega é a mesma para todos os squads, e segue exatamente o caminho desta semana:

1. Uma **camada Bronze** em `etl_northwind/bronze/`, sem filtro nenhum.
2. Uma **camada Prata** em `etl_northwind/prata/`, com as regras de limpeza **documentadas** numa célula de texto — cada regra com a justificativa de negócio ao lado.
3. Uma **camada Ouro** em `etl_northwind/ouro/`, agregada para responder a pergunta do squad.
4. As três camadas carregadas em **tabelas do schema `analytics`**, de forma idempotente (rodar duas vezes não pode duplicar).
5. Uma **validação** comparando origem e destino, com o resultado visível.
6. Um **painel** com pelo menos dois gráficos, lendo a camada Ouro **do banco**, salvo em `.png`.
7. **Conclusões geradas pelo código**, sem nenhum número digitado à mão.
8. Uma **apresentação de 5 minutos**, com a tela dividida: a árvore da extensão PostgreSQL mostrando as tabelas do squad, e o painel aberto. Responda: qual decisão de negócio o seu indicador permite tomar, e qual regra de limpeza foi a mais difícil de decidir.

| Squad | Consultoria fictícia | Pergunta de negócio |
|---|---|---|
| **A** | **Meridiano Log** | Quais transportadoras entregam mais rápido, e quanto custa o frete de cada uma? Junte `pedidos` com `shippers` pela coluna `ship_via`. |
| **B** | **Cardeal Retail** | Quais produtos têm alta receita mas estoque baixo (`units_in_stock`), e por isso correm risco de ruptura? Use `produtos` e `itens_pedido`. |
| **C** | **Atlas Global** | De quais países de fornecedores a Northwind mais depende, e qual a receita gerada por cada um? Junte `suppliers`, `produtos` e `itens_pedido`. |
| **D** | **Vetor Pessoas** | Qual o desempenho de cada vendedor em receita, número de pedidos e tempo médio de entrega? Junte `pedidos` com `employees` pela coluna `employee_id`. |
| **E** | **Prisma Insights** | Como a receita se distribui entre as categorias ao longo dos trimestres — alguma categoria cresce enquanto outra cai? Use a camada Prata com `.dt.to_period("Q")`. |

**Atenção a um detalhe que vale ponto:** as tabelas `shippers`, `suppliers` e `employees` ficaram com os nomes **em inglês** — só `clientes`, `pedidos`, `itens_pedido`, `produtos` e `categorias` foram traduzidas na Semana 11. O squad precisa lidar com essa mistura, e uma boa entrega padroniza os nomes já na camada Prata, usando `AS` na consulta ou `.rename()` no Pandas. Para não misturar as tabelas dos squads, use o nome do squad no início de cada tabela, por exemplo `analytics.squad_a_prata_fretes`.

---

### ✅ O que você construiu nesta semana

Três dias, um pipeline completo, e uma entrega de analista. Em números: **2155 linhas brutas → 1784 linhas limpas → 41 linhas de indicador**, seis tabelas no schema `analytics`, arquivos em disco, um painel e um relatório.

**Dia 1 — Extract e Bronze**
- Ambiente preparado (kernel, bibliotecas, senha) e conexão com `psycopg2`, incluindo o erro real de senha.
- Catálogo do banco pelo `information_schema`, e o conceito de **grão** (830 pedidos × 2155 itens).
- Extração com `JOIN` de 5 tabelas, inspeção com `head()`, `info()`, `describe()`, `isna()` e `duplicated()`, e a **linhagem** de cada coluna.

**Dia 2 — Transform e Prata**
- Conversão de datas com `pd.to_datetime()`, depois de ver o erro real de subtrair texto.
- Duas decisões opostas sobre valores vazios: `dropna(subset=...)` para o que ainda não aconteceu, `fillna()` para o que não se aplica.
- Regra de negócio com filtro booleano, imprecisão de ponto flutuante resolvida com `.round(2)`, e três colunas derivadas.

**Dia 3 — Ouro, Load, validação e entrega**
- Agregação com `groupby()` + `agg()` e gráficos com `matplotlib`.
- Carga no schema `analytics` com `execute_values()`, **idempotência** com `TRUNCATE`, `rollback()` na carga que falha, validação origem × destino e log de execução.
- O medalhão completo no banco, com a tradução de `NaN` para `NULL`, e rastreabilidade da Ouro até a Prata com SQL.
- A extensão **PostgreSQL** instalada e conectada no VS Code, com a árvore do banco ao lado do notebook e o editor de consultas: executar, ler resultado, errar, salvar, reabrir e recuperar pelo histórico.
- A **entrega do analista**: quatro gráficos respondendo perguntas de negócio, um painel executivo em `.png` e conclusões escritas pelo próprio código.

**A ideia que atravessa a semana inteira:** um ETL não termina quando o código roda sem erro. Ele termina quando você **provou** que o dado que chegou ao destino é o dado certo, deixou registrado como ele chegou lá — e transformou esse dado numa resposta que alguém usa para decidir.

📚 **Documentação oficial:** Pandas em https://pandas.pydata.org/docs/ · Matplotlib em https://matplotlib.org/ · psycopg2 em https://www.psycopg.org/docs/ · PostgreSQL em https://www.postgresql.org/docs/

---

*Prof. Especialista Cláudio F. Neves*